# Submission 19

In [1]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

Looking in links: /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/arc_agi-0.9.6-py3-none-any.whl
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/arcengine-0.9.3-py3-none-any.whl (from arc-agi)
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/pillow-12.1.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (from arc-agi)
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.1.1 which is incompatib

In [2]:

from __future__ import annotations

import os
import sys
import time
import math
import json
import random
import hashlib
import inspect
import pkgutil
import importlib
import subprocess
import traceback
import warnings
import copy
import re
import logging
import importlib.util
import heapq
from pathlib import Path
from collections import defaultdict, deque
from dataclasses import dataclass, field, is_dataclass, asdict
from typing import Any, Optional

import numpy as np
import pandas as pd
import requests

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

WORKDIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
SUBMISSION_PATH = WORKDIR / "submission.parquet"

# Early placeholder so Kaggle always sees the file, even if something fails later.
pd.DataFrame(
    [{"row_id": "0_0", "game_id": "0", "end_of_game": False, "score": 0.0}]
).to_parquet(SUBMISSION_PATH, index=False)


def _first_existing(paths: list[Path]) -> Optional[Path]:
    for p in paths:
        if p.exists():
            return p
    return None


def find_competition_root() -> Optional[Path]:
    direct = _first_existing(
        [
            Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3"),
            Path("/kaggle/input/arc-prize-2026-arc-agi-3"),
        ]
    )
    if direct is not None:
        return direct

    root = Path("/kaggle/input")
    if root.exists():
        for wheels in root.rglob("arc_agi_3_wheels"):
            return wheels.parent
    return None


COMP_ROOT = find_competition_root()
if COMP_ROOT is None:
    raise FileNotFoundError("Could not locate ARC-AGI-3 competition input directory.")

WHEELS_DIR = COMP_ROOT / "arc_agi_3_wheels"
ENVIRONMENTS_DIR = COMP_ROOT / "environment_files"
AGENTS_DIR = COMP_ROOT / "ARC-AGI-3-Agents"

if not WHEELS_DIR.exists():
    raise FileNotFoundError(f"Could not locate wheels directory: {WHEELS_DIR}")


def install_local_wheels(wheels_dir: Path) -> None:
    wheel_files = sorted(str(p) for p in wheels_dir.glob("*.whl"))
    if not wheel_files:
        raise FileNotFoundError(f"No wheel files found in {wheels_dir}")
    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-index",
        "--find-links",
        str(wheels_dir),
        *wheel_files,
    ]
    subprocess.check_call(cmd)


# Packages are installed in the first notebook cell as required by the competition.


def find_symbol(package_name: str, symbol_name: str) -> Any:
    package = importlib.import_module(package_name)
    if hasattr(package, symbol_name):
        return getattr(package, symbol_name)
    if hasattr(package, "__path__"):
        for modinfo in pkgutil.walk_packages(package.__path__, package.__name__ + "."):
            try:
                module = importlib.import_module(modinfo.name)
            except Exception:
                continue
            if hasattr(module, symbol_name):
                return getattr(module, symbol_name)
    raise AttributeError(f"Could not find symbol '{symbol_name}' inside package '{package_name}'")


Arcade = find_symbol("arc_agi", "Arcade")
OperationMode = find_symbol("arc_agi", "OperationMode")
try:
    GameAction = find_symbol("arcengine", "GameAction")
    GameState = find_symbol("arcengine", "GameState")
except Exception:
    GameAction = find_symbol("arc_agi", "GameAction")
    GameState = find_symbol("arc_agi", "GameState")


def resolve_mode(prefer_competition: bool) -> Any:
    members = getattr(OperationMode, "__members__", {})
    ordered_names: list[str] = []
    if prefer_competition:
        ordered_names.extend(["COMPETITION", "COMPETITON", "ONLINE", "NORMAL", "LOCAL", "OFFLINE"])
    else:
        ordered_names.extend(["OFFLINE", "NORMAL", "LOCAL", "ONLINE", "COMPETITION", "COMPETITON"])
    for name in ordered_names:
        if name in members:
            return members[name]
    try:
        members_list = list(OperationMode)
        if members_list:
            return members_list[0]
    except Exception:
        pass
    return "online" if prefer_competition else "offline"


ACTION_MEMBERS = getattr(GameAction, "__members__", {})
RESET_ACTION = ACTION_MEMBERS.get("RESET", "RESET")
ACTION_ID_TO_ENUM: dict[int, Any] = {}
for i in range(1, 8):
    ACTION_ID_TO_ENUM[i] = ACTION_MEMBERS.get(f"ACTION{i}", f"ACTION{i}")

IS_RERUN = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))

os.environ.setdefault("SCHEME", "http")
os.environ.setdefault("HOST", "gateway")
os.environ.setdefault("PORT", "8001")
os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
os.environ.setdefault("ARC_API_KEY", "test-key-123")
os.environ.setdefault("OPERATION_MODE", "COMPETITION" if IS_RERUN else "OFFLINE")

ROOT_URL = os.environ.get("ARC_BASE_URL", "http://gateway:8001/").rstrip("/")
HEADERS = {
    "X-API-Key": os.environ.get("ARC_API_KEY", ""),
    "Accept": "application/json",
}


def wait_for_gateway(base_url: str, headers: dict[str, str], timeout_s: int = 180) -> bool:
    if not IS_RERUN:
        return False
    deadline = time.time() + timeout_s
    last_err = None
    url = base_url.rstrip("/") + "/api/games"
    while time.time() < deadline:
        try:
            r = requests.get(url, headers=headers, timeout=5)
            if r.ok:
                return True
            last_err = f"{r.status_code}: {r.text[:200]}"
        except Exception as exc:
            last_err = repr(exc)
        time.sleep(2)
    print(f"Gateway unavailable, falling back to local environments when possible: {last_err}")
    return False


GATEWAY_READY = wait_for_gateway(ROOT_URL, HEADERS, timeout_s=600)


def make_arcade() -> Any:
    prefer_competition = IS_RERUN and GATEWAY_READY
    kwargs: dict[str, Any] = {}
    sig = inspect.signature(Arcade)
    op_mode = resolve_mode(prefer_competition)
    if "operation_mode" in sig.parameters:
        kwargs["operation_mode"] = op_mode
    if "arc_api_key" in sig.parameters and IS_RERUN:
        kwargs["arc_api_key"] = os.environ.get("ARC_API_KEY", "")
    if "arc_base_url" in sig.parameters and IS_RERUN:
        kwargs["arc_base_url"] = os.environ.get("ARC_BASE_URL", ROOT_URL)
    if "environments_dir" in sig.parameters and ENVIRONMENTS_DIR.exists():
        kwargs["environments_dir"] = str(ENVIRONMENTS_DIR)
    return Arcade(**kwargs)


def maybe_get(obj: Any, *names: str) -> Any:
    for name in names:
        if isinstance(obj, dict) and name in obj:
            return obj[name]
        if hasattr(obj, name):
            return getattr(obj, name)
    return None


def deep_find(obj: Any, names: tuple[str, ...], max_depth: int = 3) -> Any:
    seen: set[int] = set()

    def rec(x: Any, depth: int) -> Any:
        if depth < 0 or x is None:
            return None
        try:
            oid = id(x)
            if oid in seen:
                return None
            seen.add(oid)
        except Exception:
            pass

        got = maybe_get(x, *names)
        if got is not None:
            return got

        if isinstance(x, dict):
            for v in x.values():
                found = rec(v, depth - 1)
                if found is not None:
                    return found
        elif isinstance(x, (list, tuple)):
            for v in x:
                found = rec(v, depth - 1)
                if found is not None:
                    return found
        else:
            d = getattr(x, "__dict__", None)
            if isinstance(d, dict):
                for v in d.values():
                    found = rec(v, depth - 1)
                    if found is not None:
                        return found
        return None

    return rec(obj, max_depth)


def to_plain(obj: Any, depth: int = 6) -> Any:
    if depth < 0:
        return None
    if obj is None or isinstance(obj, (str, int, float, bool)):
        return obj
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, dict):
        return {k: to_plain(v, depth - 1) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [to_plain(v, depth - 1) for v in obj]
    if is_dataclass(obj):
        return to_plain(asdict(obj), depth - 1)
    if hasattr(obj, "model_dump"):
        try:
            return to_plain(obj.model_dump(), depth - 1)
        except Exception:
            pass
    if hasattr(obj, "dict"):
        try:
            return to_plain(obj.dict(), depth - 1)
        except Exception:
            pass
    if hasattr(obj, "__dict__"):
        try:
            return {k: to_plain(v, depth - 1) for k, v in vars(obj).items() if not k.startswith("_")}
        except Exception:
            pass
    return repr(obj)


def normalize_action_id(action: Any) -> Optional[int]:
    if action is None:
        return None
    if isinstance(action, (int, np.integer)):
        return int(action)
    name = None
    if isinstance(action, str):
        name = action
    elif hasattr(action, "name"):
        name = str(action.name)
    elif hasattr(action, "value") and isinstance(action.value, str):
        name = action.value
    if name is not None:
        name = name.split(".")[-1].upper()
        if name == "RESET":
            return 0
        if name.startswith("ACTION"):
            try:
                return int(name.replace("ACTION", "").replace("_", ""))
            except Exception:
                return None
    if hasattr(action, "value") and isinstance(action.value, (int, np.integer)):
        return int(action.value)
    return None


def unique_preserve_order(items: list[Any]) -> list[Any]:
    out = []
    seen = set()
    for item in items:
        key = repr(item)
        if key in seen:
            continue
        seen.add(key)
        out.append(item)
    return out


def unwrap_observation(step_output: Any) -> Any:
    out = step_output
    if isinstance(out, tuple) and len(out) > 0:
        out = out[0]
    frame_data = maybe_get(out, "frame_data", "observation")
    if frame_data is not None:
        if deep_find(frame_data, ("frame", "frames", "grid", "grids"), 2) is not None:
            out = frame_data
    return out


def state_name(obs: Any) -> str:
    st = deep_find(obs, ("state", "game_state"), 2)
    if st is None:
        return ""
    if isinstance(st, str):
        return st.split(".")[-1].upper()
    if hasattr(st, "name"):
        return str(st.name).upper()
    return str(st).split(".")[-1].upper()


def levels_completed(obs: Any) -> int:
    val = deep_find(obs, ("levels_completed", "score"), 2)
    if val is None:
        return 0
    try:
        return int(val)
    except Exception:
        try:
            return int(float(val))
        except Exception:
            return 0


def extract_available_action_ids(obs: Any, env: Any = None) -> set[int]:
    raw = deep_find(obs, ("available_actions", "actions"), 2)
    if raw is None and env is not None:
        raw = maybe_get(env, "available_actions", "action_space")
        if raw is not None:
            raw = maybe_get(raw, "available_actions") or raw

    if raw is None:
        return {i for i in range(1, 8) if ACTION_ID_TO_ENUM.get(i) is not None}

    if isinstance(raw, dict):
        raw = raw.keys()

    ids: set[int] = set()
    try:
        iterable = list(raw)
    except Exception:
        iterable = []

    for a in iterable:
        aid = normalize_action_id(a)
        if aid is not None and aid != 0:
            ids.add(aid)

    if not ids:
        ids = {i for i in range(1, 8) if ACTION_ID_TO_ENUM.get(i) is not None}
    return ids


def extract_latest_frame(obs: Any) -> tuple[np.ndarray, int]:
    frame_obj = deep_find(obs, ("frame", "frames", "grid", "grids"), 3)
    if frame_obj is None:
        raise ValueError("Could not extract frame/grid from observation.")
    arr = np.asarray(frame_obj, dtype=np.uint8)
    if arr.ndim == 3:
        return arr[-1].copy(), int(arr.shape[0])
    if arr.ndim == 2:
        return arr.copy(), 1
    if arr.ndim == 1:
        side = int(round(math.sqrt(arr.size)))
        if side * side == arr.size:
            return arr.reshape(side, side).astype(np.uint8), 1
    raise ValueError(f"Unexpected frame shape: {arr.shape}")


def step_env(env: Any, action_token: Any, data: Optional[dict[str, int]] = None, reasoning: Optional[str] = None) -> Any:
    patterns = [
        ((action_token,), {"data": data, "reasoning": reasoning}),
        ((action_token,), {"data": data}),
        ((action_token,), {"reasoning": reasoning}),
        ((action_token,), {}),
    ]
    if data is not None:
        patterns.extend(
            [
                ((action_token, data, reasoning), {}),
                ((action_token, data), {}),
            ]
        )

    last_exc = None
    for args, kwargs in patterns:
        try:
            return env.step(*args, **{k: v for k, v in kwargs.items() if v is not None})
        except TypeError as exc:
            last_exc = exc
        except Exception:
            raise
    raise last_exc or RuntimeError("Failed to call env.step")


def take_action(env: Any, action_id: int, data: Optional[dict[str, int]] = None, reasoning: Optional[str] = None) -> Any:
    action_enum = RESET_ACTION if action_id == 0 else ACTION_ID_TO_ENUM.get(action_id, action_id)
    tokens = []
    if action_enum is not None:
        tokens.append(action_enum)
        name = getattr(action_enum, "name", None)
        if name:
            tokens.append(name)
        value = getattr(action_enum, "value", None)
        if value is not None:
            tokens.append(value)
    tokens.append(action_id)
    tokens = unique_preserve_order(tokens)

    last_exc = None
    for tok in tokens:
        try:
            out = step_env(env, tok, data=data, reasoning=reasoning)
            return unwrap_observation(out)
        except TypeError as exc:
            last_exc = exc
        except Exception:
            raise
    raise last_exc or RuntimeError(f"Could not step environment with action {action_id}")


def perform_reset(env: Any, reason: str = "reset", max_attempts: int = 3) -> tuple[Any, int]:
    obs = None
    used = 0
    for _ in range(max_attempts):
        obs = take_action(env, 0, data=None, reasoning=reason)
        used += 1
        if state_name(obs) not in {"NOT_PLAYED", ""}:
            break
    return obs, used


def normalize_game_ids(items: Any) -> list[str]:
    ids: list[str] = []
    for item in items or []:
        gid = None
        if isinstance(item, str):
            gid = item
        elif isinstance(item, dict):
            gid = item.get("game_id") or item.get("id")
        else:
            gid = getattr(item, "game_id", None) or getattr(item, "id", None)
            if gid is None:
                text = str(item)
                if text and text != object.__repr__(item):
                    gid = text
        if gid is not None:
            ids.append(str(gid))
    return ids


INFINITY = np.iinfo(np.int32).max
EDGE_DTYPE = np.dtype(
    [
        ("group", "i4"),
        ("result", "i4"),
        ("target", "U64"),
        ("distance", "i4"),
        ("errors", "i4"),
    ]
)


@dataclass
class NodeInfo:
    name: str
    total_candidates: int
    num_groups: int
    group2remaining_candidate_ids: list[set[int]]
    error_threshold: int = 3
    closed: bool = False
    distance: int = INFINITY
    edge_data: np.ndarray = field(init=False)

    def __post_init__(self) -> None:
        self.group2remaining_candidate_ids = [set(s) for s in self.group2remaining_candidate_ids]
        self.edge_data = np.zeros(self.total_candidates, dtype=EDGE_DTYPE)
        for group_id, candidate_ids in enumerate(self.group2remaining_candidate_ids):
            if candidate_ids:
                self.edge_data["group"][list(candidate_ids)] = group_id

    def has_open_group(self, group_id: int) -> bool:
        for gid in range(min(group_id, self.num_groups - 1) + 1):
            if self.group2remaining_candidate_ids[gid]:
                return True
        return False

    def record_test(self, edge_idx: int, success: int, target_node: Optional[str] = None) -> bool:
        if edge_idx < 0 or edge_idx >= self.total_candidates:
            return False

        current_result = int(self.edge_data["result"][edge_idx])
        if current_result != 0:
            if success == 1 and target_node and str(target_node) != str(self.edge_data["target"][edge_idx]):
                # keep the shorter path if the node is being repaired later
                pass
            else:
                return False

        edge_group_id = int(self.edge_data["group"][edge_idx])

        if success == -1:
            self.edge_data["errors"][edge_idx] += 1
            if self.edge_data["errors"][edge_idx] < self.error_threshold:
                return False
            self.edge_data["errors"][edge_idx] = 0
            new_group_id = edge_group_id + 1
            self.group2remaining_candidate_ids[edge_group_id].discard(edge_idx)
            if new_group_id <= self.num_groups - 1:
                self.edge_data["group"][edge_idx] = new_group_id
                self.group2remaining_candidate_ids[new_group_id].add(edge_idx)
                return False
            self.edge_data["result"][edge_idx] = -1
            self.edge_data["distance"][edge_idx] = INFINITY
            return True

        self.group2remaining_candidate_ids[edge_group_id].discard(edge_idx)

        if success == 1:
            self.edge_data["result"][edge_idx] = 1
            self.edge_data["target"][edge_idx] = str(target_node or "")
            self.edge_data["distance"][edge_idx] = -1
        else:
            self.edge_data["result"][edge_idx] = -1
            self.edge_data["distance"][edge_idx] = INFINITY
        return True


class GraphExplorer:
    def __init__(self, n_groups: int = 5, verbose: int = 0):
        self._n_groups = max(1, int(n_groups))
        self._verbose = verbose
        self.reset()

    def reset(self) -> None:
        self._nodes: dict[str, NodeInfo] = {}
        self._G: dict[str, set[tuple[int, str]]] = defaultdict(set)
        self._G_rev: dict[str, set[tuple[int, str]]] = defaultdict(set)
        self._frontier: set[str] = set()
        self._dist: dict[str, int] = {}
        self._next: dict[str, tuple[int, str]] = {}
        self._active_group = 0
        self._empty = True
        self.suspicious_transitions: dict[tuple[str, int, str], int] = {}
        self.suspicious_transitions_threshold = 3

    @property
    def active_group(self) -> int:
        return self._active_group

    @property
    def frontier(self) -> set[str]:
        return self._frontier

    def distance(self, node: str) -> int:
        return int(self._dist.get(node, INFINITY))

    def has_node(self, node: str) -> bool:
        return node in self._nodes

    def initialize(self, start_node: str, num_candidates: int, group2remaining_candidate_ids: list[set[int]]) -> None:
        self._add_new_node(start_node, num_candidates, group2remaining_candidate_ids)

    def _add_new_node(self, node: str, n_candidates: int, group2remaining_candidate_ids: list[set[int]]) -> None:
        if node in self._nodes:
            return
        self._nodes[node] = NodeInfo(
            name=node,
            total_candidates=n_candidates,
            num_groups=self._n_groups,
            group2remaining_candidate_ids=group2remaining_candidate_ids,
        )
        self._G[node] = set()
        self._G_rev[node] = set()
        self._empty = False
        if self._nodes[node].has_open_group(self._active_group):
            self._frontier.add(node)
        else:
            self._close_node(node)

    def _remove_edge(self, node: str, edge_idx: int) -> None:
        for existing_idx, target in list(self._G.get(node, set())):
            if int(existing_idx) == int(edge_idx):
                self._G[node].discard((existing_idx, target))
                self._G_rev[target].discard((existing_idx, node))


    def _close_node(self, node: str) -> None:
        info = self._nodes[node]
        if info.closed:
            return
        info.closed = True
        self._frontier.discard(node)
        self._rebuild_distances()

    def _rebuild_distances(self) -> None:
        self._dist.clear()
        self._next.clear()

        dq = deque()
        for node, info in self._nodes.items():
            info.distance = INFINITY
            self._dist[node] = INFINITY

        for node in self._frontier:
            self._dist[node] = 0
            self._nodes[node].distance = 0
            dq.append(node)

        while dq:
            v = dq.popleft()
            v_dist = self._dist.get(v, INFINITY)
            for edge_idx, u in self._G_rev.get(v, ()):
                cand_dist = v_dist + 1
                self._nodes[u].edge_data["distance"][edge_idx] = cand_dist
                if cand_dist < self._dist.get(u, INFINITY):
                    self._dist[u] = cand_dist
                    self._nodes[u].distance = cand_dist
                    self._next[u] = (edge_idx, v)
                    dq.append(u)

    def _maybe_advance_group(self, current_node: str) -> None:
        while self.distance(current_node) >= INFINITY and self._active_group < self._n_groups - 1:
            self._active_group += 1
            self._frontier.clear()
            for node, info in self._nodes.items():
                info.closed = False
                if info.has_open_group(self._active_group):
                    self._frontier.add(node)
            self._rebuild_distances()

    def record_test(
        self,
        node: str,
        edge_idx: int,
        success: int,
        target_node: Optional[str] = None,
        target_num_candidates: Optional[int] = None,
        group2remaining_candidate_ids: Optional[list[set[int]]] = None,
        suspicious_transition: bool = False,
    ) -> None:
        if node not in self._nodes:
            return

        if suspicious_transition and success == 1 and target_node is not None:
            key = (node, edge_idx, target_node)
            self.suspicious_transitions[key] = self.suspicious_transitions.get(key, 0) + 1
            if self.suspicious_transitions[key] < self.suspicious_transitions_threshold:
                return

        node_info = self._nodes[node]
        wrote = node_info.record_test(edge_idx, int(success), target_node)
        if not wrote:
            return

        if success == 1 and target_node is not None:
            self._remove_edge(node, edge_idx)
            if target_node not in self._nodes:
                if target_num_candidates is None or group2remaining_candidate_ids is None:
                    return
                self._add_new_node(target_node, target_num_candidates, group2remaining_candidate_ids)
            self._G[node].add((edge_idx, target_node))
            self._G_rev[target_node].add((edge_idx, node))

            if not self._nodes[node].has_open_group(self._active_group):
                self._close_node(node)

            if self._nodes[target_node].has_open_group(self._active_group):
                self._rebuild_distances()
            else:
                self._close_node(target_node)
                self._maybe_advance_group(target_node)
        else:
            self._remove_edge(node, edge_idx)
            if not self._nodes[node].has_open_group(self._active_group):
                self._close_node(node)
                self._maybe_advance_group(node)

    def open_candidate_ids(self, node: str) -> list[int]:
        if node not in self._nodes:
            return []
        node_info = self._nodes[node]
        if not node_info.has_open_group(self._active_group):
            return []
        out: list[int] = []
        for gid in range(self._active_group + 1):
            out.extend(sorted(node_info.group2remaining_candidate_ids[gid]))
        return out

    def successful_candidate_ids(self, node: str) -> list[int]:
        if node not in self._nodes:
            return []
        node_info = self._nodes[node]
        lowest_dist = self.distance(node)
        if lowest_dist >= INFINITY:
            return []
        return [
            idx
            for idx, row in enumerate(node_info.edge_data)
            if int(row["result"]) == 1
            and int(row["group"]) <= self._active_group
            and int(row["distance"]) <= lowest_dist
        ]

    def edge_status(self, node: str, edge_idx: int) -> int:
        if node not in self._nodes:
            return 0
        node_info = self._nodes[node]
        if edge_idx < 0 or edge_idx >= node_info.total_candidates:
            return 0
        return int(node_info.edge_data["result"][edge_idx])

    def edge_distance(self, node: str, edge_idx: int) -> int:
        if node not in self._nodes:
            return INFINITY
        node_info = self._nodes[node]
        if edge_idx < 0 or edge_idx >= node_info.total_candidates:
            return INFINITY
        return int(node_info.edge_data["distance"][edge_idx])

    def choose_edge(self, node: str) -> Optional[int]:
        open_ids = self.open_candidate_ids(node)
        if open_ids:
            return random.choice(open_ids)
        options = self.successful_candidate_ids(node)
        if options:
            return random.choice(options)
        return None


@dataclass
class BetaCounter:
    success: float = 1.0
    failure: float = 1.0

    def mean(self) -> float:
        denom = self.success + self.failure
        return float(self.success / denom) if denom > 0 else 0.5

    @property
    def count(self) -> float:
        return max(0.0, self.success + self.failure - 2.0)

    def update(self, success_inc: float, failure_inc: float) -> None:
        self.success += max(0.0, float(success_inc))
        self.failure += max(0.0, float(failure_inc))


class FeatureMemory:
    def __init__(self) -> None:
        self._stats: dict[tuple[Any, ...], BetaCounter] = {}
        self.total_updates = 0.0

    def _get(self, key: tuple[Any, ...]) -> BetaCounter:
        if key not in self._stats:
            self._stats[key] = BetaCounter()
        return self._stats[key]

    def mean(self, key: tuple[Any, ...]) -> float:
        return self._get(key).mean()

    def count(self, key: tuple[Any, ...]) -> float:
        return self._get(key).count

    def update(self, key: tuple[Any, ...], success_inc: float, failure_inc: float) -> None:
        self._get(key).update(success_inc, failure_inc)
        self.total_updates += max(0.0, float(success_inc)) + max(0.0, float(failure_inc))

    def _candidate_weighted_keys(self, candidate: dict[str, Any]) -> list[tuple[tuple[Any, ...], float]]:
        feats = candidate.get("features") or {}
        kind = str(candidate.get("kind", ""))
        action_id = int(candidate.get("action_id", -1))

        if kind == "simple":
            return [
                (("kind", "simple"), 0.35),
                (("aid", action_id), 0.65),
            ]

        if kind == "click":
            source = str(feats.get("source", "click"))
            group = int(candidate.get("group", feats.get("group", 4)))
            color = int(feats.get("color", -1))
            area_bucket = int(feats.get("area_bucket", -1))
            rect = int(feats.get("rect", 0))
            edge_mask = int(feats.get("edge_mask", 0))
            rep_idx = int(feats.get("rep_idx", -1))
            bin_x = int(feats.get("bin_x", -1))
            bin_y = int(feats.get("bin_y", -1))
            return [
                (("kind", "click"), 0.10),
                (("aid", action_id), 0.18),
                (("group", group), 0.18),
                (("source", source), 0.08),
                (("color", color), 0.12),
                (("group_color", group, color), 0.10),
                (("bin", bin_x, bin_y), 0.10),
                (("rep", rep_idx), 0.04),
                (("area", area_bucket), 0.04),
                (("rect", rect), 0.03),
                (("edge", edge_mask), 0.03),
            ]

        return [(("kind", kind or "other"), 1.0)]

    def estimate(self, candidate: dict[str, Any]) -> float:
        weighted = self._candidate_weighted_keys(candidate)
        if not weighted:
            return 0.5
        total_w = 0.0
        total_v = 0.0
        for key, w in weighted:
            total_w += float(w)
            total_v += float(w) * self.mean(key)
        return float(total_v / total_w) if total_w > 0 else 0.5

    def exposure(self, candidate: dict[str, Any]) -> float:
        weighted = self._candidate_weighted_keys(candidate)
        if not weighted:
            return 0.0
        total_w = 0.0
        total_v = 0.0
        for key, w in weighted:
            total_w += float(w)
            total_v += float(w) * self.count(key)
        return float(total_v / total_w) if total_w > 0 else 0.0

    def update_candidate(
        self,
        candidate: dict[str, Any],
        *,
        changed: bool,
        level_up: bool = False,
        game_over: bool = False,
        errored: bool = False,
        suspicious: bool = False,
    ) -> None:
        if candidate.get("kind") == "meta_reset":
            return

        if level_up:
            success_inc = 1.20
            failure_inc = 0.00
        elif changed and not suspicious:
            success_inc = 0.65
            failure_inc = 0.35
        elif changed and suspicious:
            success_inc = 0.20
            failure_inc = 0.80
        elif game_over or errored:
            success_inc = 0.00
            failure_inc = 1.25
        else:
            success_inc = 0.00
            failure_inc = 1.00

        for key, _ in self._candidate_weighted_keys(candidate):
            self.update(key, success_inc, failure_inc)

    def kind_bias(self) -> float:
        return float(self.mean(("kind", "click")) - self.mean(("kind", "simple")))


GLOBAL_FEATURE_MEMORY = FeatureMemory()


@dataclass
class FrameView:
    node_id: str
    hash_id: str
    raw_frame: np.ndarray
    masked_frame_for_segmentation: np.ndarray
    hash_frame_input: np.ndarray
    num_frames: int
    state: str
    levels_completed: int
    available_action_ids: set[int]
    undo_available: bool
    segments: list[dict[str, Any]]
    candidates: list[dict[str, Any]]
    groups: list[set[int]]


class FrameProcessor:
    OFFSETS4 = ((-1, 0), (1, 0), (0, -1), (0, 1))

    def __init__(self) -> None:
        self.frame_shape = (64, 64)
        self.status_bar_color = 16
        self.salient_colors = {6, 7, 8, 9, 10, 11, 12, 13, 14, 15}
        self.non_salient_colors = {0, 1, 2, 3, 4, 5}
        self.status_bar_ratio_threshold = 5.0
        self.status_bar_twins_threshold = 3

    def _set_shape(self, shape: tuple[int, int]) -> None:
        self.frame_shape = tuple(int(v) for v in shape)

    @property
    def status_bar_distance_threshold(self) -> int:
        return max(1, min(3, min(self.frame_shape) // 12 + 1))

    @property
    def minimal_width(self) -> int:
        return 2 if min(self.frame_shape) >= 8 else 1

    @property
    def maximal_width(self) -> int:
        return max(2, min(self.frame_shape) // 2)

    def segment_frame(self, frame: np.ndarray) -> tuple[np.ndarray, list[dict[str, Any]]]:
        frame = np.asarray(frame, dtype=np.uint8)
        self._set_shape(tuple(frame.shape))
        h, w = frame.shape
        label_map = np.full((h, w), -1, dtype=np.int32)
        components: list[dict[str, Any]] = []
        cid = -1

        for y in range(h):
            for x in range(w):
                if label_map[y, x] != -1:
                    continue
                cid += 1
                color = int(frame[y, x])
                q = deque([(y, x)])
                label_map[y, x] = cid

                min_x = max_x = x
                min_y = max_y = y
                area = 0
                points: list[tuple[int, int]] = []

                while q:
                    cy, cx = q.popleft()
                    points.append((cy, cx))
                    area += 1
                    min_x = min(min_x, cx)
                    max_x = max(max_x, cx)
                    min_y = min(min_y, cy)
                    max_y = max(max_y, cy)
                    for dy, dx in self.OFFSETS4:
                        ny, nx = cy + dy, cx + dx
                        if 0 <= ny < h and 0 <= nx < w and label_map[ny, nx] == -1 and int(frame[ny, nx]) == color:
                            label_map[ny, nx] = cid
                            q.append((ny, nx))

                rect_area = (max_x - min_x + 1) * (max_y - min_y + 1)
                components.append(
                    {
                        "bounding_box": (min_x, min_y, max_x, max_y),
                        "color": color,
                        "area": area,
                        "is_rectangle": area == rect_area,
                        "points": points,
                    }
                )

        buckets: dict[tuple[int, bool, int], list[int]] = defaultdict(list)
        for i, comp in enumerate(components):
            key = (int(comp["area"]), bool(comp["is_rectangle"]), int(comp["color"]))
            buckets[key].append(i)

        for i, comp in enumerate(components):
            key = (int(comp["area"]), bool(comp["is_rectangle"]), int(comp["color"]))
            twins = [j for j in buckets[key] if j != i]
            comp["number_of_twins"] = len(twins)
            comp["twin_ids"] = twins

        return label_map, components

    def check_segment_fully_on_edge(self, segment: dict[str, Any], edges: Optional[list[str]] = None) -> list[str]:
        x1, y1, x2, y2 = segment["bounding_box"]
        thr = self.status_bar_distance_threshold
        if edges is None:
            edges = ["any"]
        result = []
        if "left" in edges or "any" in edges:
            if max(x1, x2) < thr:
                result.append("left")
        if "right" in edges or "any" in edges:
            if min(x1, x2) >= self.frame_shape[1] - thr:
                result.append("right")
        if "top" in edges or "any" in edges:
            if max(y1, y2) < thr:
                result.append("top")
        if "bottom" in edges or "any" in edges:
            if min(y1, y2) >= self.frame_shape[0] - thr:
                result.append("bottom")
        return result

    def check_segment_ratio(self, segment: dict[str, Any], direction: str = "any") -> bool:
        x1, y1, x2, y2 = segment["bounding_box"]
        x_len = x2 - x1 + 1
        y_len = y2 - y1 + 1
        ratio = x_len / max(y_len, 1)
        if ratio >= self.status_bar_ratio_threshold and direction in ("any", "horizontal"):
            return True
        if ratio <= 1.0 / self.status_bar_ratio_threshold and direction in ("any", "vertical"):
            return True
        return False

    def segment_twins_on_edge(self, segment: dict[str, Any], frame_segments: list[dict[str, Any]], edges: Optional[list[str]] = None) -> list[int]:
        if edges is None:
            edges = self.check_segment_fully_on_edge(segment, ["any"])
            if not edges:
                return []
        twins = []
        for twin_id in segment["twin_ids"]:
            twin_edges = self.check_segment_fully_on_edge(frame_segments[twin_id], edges)
            if twin_edges:
                twins.append(twin_id)
        return twins

    def identify_status_bars(self, segmented_frame: np.ndarray, frame_segments: list[dict[str, Any]]) -> np.ndarray:
        self._set_shape(tuple(segmented_frame.shape))
        checked: set[int] = set()
        status_segment_groups: list[list[int]] = []

        for i, segment in enumerate(frame_segments):
            if i in checked:
                continue
            checked.add(i)
            on_edge = self.check_segment_fully_on_edge(segment, ["any"])
            if not on_edge:
                continue

            dirs = []
            if "left" in on_edge or "right" in on_edge:
                dirs.append("vertical")
            if "top" in on_edge or "bottom" in on_edge:
                dirs.append("horizontal")
            direction = "any" if len(dirs) == 2 else dirs[0]

            candidate_ids = [i]
            is_long = self.check_segment_ratio(segment, direction)
            if not is_long:
                twin_ids = self.segment_twins_on_edge(segment, frame_segments)
                for twin_id in twin_ids:
                    checked.add(twin_id)
                if len(twin_ids) + 1 < self.status_bar_twins_threshold:
                    continue
                candidate_ids.extend(twin_ids)

            status_segment_groups.append(candidate_ids)

        mask = np.zeros(segmented_frame.shape, dtype=bool)
        for group in status_segment_groups:
            for seg_id in group:
                mask[segmented_frame == seg_id] = True
        return mask

    def hash_frame(self, frame: np.ndarray) -> str:
        frame = np.asarray(frame, dtype=np.uint8, order="C")
        flat = frame.ravel()
        if flat.size % 2 == 1:
            flat = np.concatenate([flat, np.zeros(1, dtype=np.uint8)])
        packed = (flat[0::2] << 4) | (flat[1::2] & 0x0F)
        return hashlib.blake2b(
            packed.tobytes(),
            digest_size=16,
            person=repr(frame.shape).encode(),
        ).hexdigest()

    def segment_group(self, segment: dict[str, Any]) -> int:
        x1, y1, x2, y2 = segment["bounding_box"]
        xw = x2 - x1 + 1
        yw = y2 - y1 + 1
        color = int(segment["color"])
        is_salient = color in self.salient_colors
        is_medium = self.minimal_width <= xw <= self.maximal_width and self.minimal_width <= yw <= self.maximal_width
        is_status_bar = color == self.status_bar_color

        if is_salient and is_medium:
            return 0
        if is_medium:
            return 1
        if is_salient:
            return 2
        if not is_status_bar:
            return 3
        return 4


    def representative_points(self, segment: dict[str, Any]) -> list[tuple[int, int]]:
        pts = segment["points"]
        if not pts:
            return []
        x1, y1, x2, y2 = segment["bounding_box"]
        cy = 0.5 * (y1 + y2)
        cx = 0.5 * (x1 + x2)

        def dist2(p: tuple[int, int], tx: float, ty: float) -> float:
            py, px = p
            return (py - ty) ** 2 + (px - tx) ** 2

        def nearest(tx: float, ty: float) -> tuple[int, int]:
            return min(pts, key=lambda p: dist2(p, tx, ty))

        center_point = nearest(cx, cy)
        first_point = min(pts)
        last_point = max(pts)

        out = [center_point]
        if len(pts) >= 12:
            out.append(first_point)
        if len(pts) >= 32:
            out.append(last_point)

        width = x2 - x1 + 1
        height = y2 - y1 + 1
        if len(pts) >= 24 and max(width, height) >= (2 * min(width, height) + 2):
            if width >= height:
                out.append(nearest(x1 + 0.25 * (x2 - x1), cy))
                out.append(nearest(x1 + 0.75 * (x2 - x1), cy))
            else:
                out.append(nearest(cx, y1 + 0.25 * (y2 - y1)))
                out.append(nearest(cx, y1 + 0.75 * (y2 - y1)))

        dedup = []
        seen = set()
        for py, px in out:
            key = (int(px), int(py))
            if key not in seen:
                seen.add(key)
                dedup.append((int(px), int(py)))
        return dedup

    def edge_mask(self, segment: dict[str, Any]) -> int:
        mask = 0
        for edge in self.check_segment_fully_on_edge(segment, ["any"]):
            if edge == "left":
                mask |= 1
            elif edge == "right":
                mask |= 2
            elif edge == "top":
                mask |= 4
            elif edge == "bottom":
                mask |= 8
        return int(mask)

    def area_bucket(self, area: int) -> int:
        return int(min(7, max(0, int(math.log2(max(1, int(area)))))))

    def candidate_features(
        self,
        *,
        kind: str,
        action_id: int,
        group: int,
        x: Optional[int] = None,
        y: Optional[int] = None,
        raw_frame: Optional[np.ndarray] = None,
        segment: Optional[dict[str, Any]] = None,
        source: str = "simple",
        rep_idx: int = 0,
    ) -> dict[str, Any]:
        out: dict[str, Any] = {
            "kind": str(kind),
            "action_id": int(action_id),
            "group": int(group),
            "source": str(source),
        }
        if kind == "simple":
            return out

        h, w = raw_frame.shape if raw_frame is not None else self.frame_shape
        xi = int(0 if x is None else x)
        yi = int(0 if y is None else y)
        out["x"] = xi
        out["y"] = yi
        out["bin_x"] = int(min(3, (4 * xi) // max(1, w - 1))) if w > 1 else 0
        out["bin_y"] = int(min(3, (4 * yi) // max(1, h - 1))) if h > 1 else 0
        out["rep_idx"] = int(rep_idx)

        if segment is not None:
            out["color"] = int(segment["color"])
            out["area_bucket"] = self.area_bucket(int(segment["area"]))
            out["rect"] = int(bool(segment["is_rectangle"]))
            out["edge_mask"] = self.edge_mask(segment)
        else:
            color = int(raw_frame[yi, xi]) if raw_frame is not None else -1
            out["color"] = color
            out["area_bucket"] = 0
            out["rect"] = 1
            edge_mask = 0
            if xi <= 1:
                edge_mask |= 1
            if xi >= w - 2:
                edge_mask |= 2
            if yi <= 1:
                edge_mask |= 4
            if yi >= h - 2:
                edge_mask |= 8
            out["edge_mask"] = int(edge_mask)
        return out

    def coarse_grid_points(self, frame: np.ndarray, invalid_mask: np.ndarray, grid_n: int = 4) -> list[tuple[int, int]]:
        h, w = frame.shape
        ys = np.linspace(0, h - 1, grid_n).round().astype(int).tolist()
        xs = np.linspace(0, w - 1, grid_n).round().astype(int).tolist()

        points: list[tuple[int, int]] = []
        seen = set()
        for y in ys:
            for x in xs:
                if not invalid_mask[y, x]:
                    key = (x, y)
                    if key not in seen:
                        seen.add(key)
                        points.append(key)
                    continue
                found = None
                for radius in range(1, 4):
                    for dy in range(-radius, radius + 1):
                        for dx in range(-radius, radius + 1):
                            ny, nx = y + dy, x + dx
                            if 0 <= ny < h and 0 <= nx < w and not invalid_mask[ny, nx]:
                                found = (nx, ny)
                                break
                        if found is not None:
                            break
                    if found is not None:
                        break
                if found is not None and found not in seen:
                    seen.add(found)
                    points.append(found)
        return points



class LevelExplorerAgent:
    INVERSE_ACTION = {1: 2, 2: 1, 3: 4, 4: 3}

    def __init__(self, game_id: str, n_groups: int = 5, global_memory: Optional[FeatureMemory] = None) -> None:
        self.game_id = str(game_id)
        self.n_groups = n_groups
        self.fp = FrameProcessor()
        self.graph = GraphExplorer(n_groups=n_groups)
        self.global_memory = global_memory if global_memory is not None else GLOBAL_FEATURE_MEMORY
        seed_bytes = hashlib.blake2b(self.game_id.encode(), digest_size=4).digest()
        seed_int = int.from_bytes(seed_bytes, "big") ^ SEED
        self.py_rng = random.Random(seed_int)
        self.np_rng = np.random.default_rng(seed_int)
        self.reset_level_state()

    def reset_level_state(self) -> None:
        self.level_memory = FeatureMemory()
        self.status_bar_mask: Optional[np.ndarray] = None
        self.start_node: Optional[str] = None
        self.view_cache: dict[str, FrameView] = {}
        self.graph.reset()
        self.steps_in_level = 0
        self.resets_in_level = 0
        self.steps_since_new_node = 0
        self.steps_since_hash_change = 0
        self.last_unique_nodes = 0
        self.undo_stack: list[str] = []
        self.last_changed_action_id: Optional[int] = None

    def remember(
        self,
        candidate: dict[str, Any],
        *,
        changed: bool,
        level_up: bool = False,
        game_over: bool = False,
        errored: bool = False,
        suspicious: bool = False,
    ) -> None:
        if candidate.get("kind") in {"meta_reset", "undo"}:
            return
        self.level_memory.update_candidate(
            candidate,
            changed=changed,
            level_up=level_up,
            game_over=game_over,
            errored=errored,
            suspicious=suspicious,
        )
        self.global_memory.update_candidate(
            candidate,
            changed=changed,
            level_up=level_up,
            game_over=game_over,
            errored=errored,
            suspicious=suspicious,
        )

    def candidate_priority(self, candidate: dict[str, Any], *, path_mode: bool = False, distance: int = 0) -> float:
        kind = str(candidate.get("kind", ""))
        aid = int(candidate.get("action_id", -1))

        if kind == "undo":
            score = -0.35 - 0.06 * float(distance)
            score += 1e-6 * self.py_rng.random()
            return float(score)

        local_est = self.level_memory.estimate(candidate)
        global_est = self.global_memory.estimate(candidate)
        prior = 0.72 * local_est + 0.28 * global_est

        local_exposure = self.level_memory.exposure(candidate)
        global_exposure = min(16.0, self.global_memory.exposure(candidate))
        exposure = 0.82 * local_exposure + 0.18 * global_exposure
        novelty = 1.0 / math.sqrt(1.0 + exposure)

        group = int(candidate.get("group", 0))
        feats = candidate.get("features") or {}
        source = str(feats.get("source", ""))

        if path_mode:
            score = 1.62 * prior + 0.07 * novelty - 0.21 * float(distance)
        else:
            score = 1.32 * prior + 0.13 * novelty - 0.20 * float(group)

        kind_bias = 0.60 * self.level_memory.kind_bias() + 0.20 * self.global_memory.kind_bias()
        if kind == "click":
            score += 0.08 * kind_bias
            if source == "segment":
                score += 0.03
            color = int(feats.get("color", -1))
            if color in self.fp.salient_colors:
                score += 0.04
        elif kind == "simple":
            score -= 0.05 * kind_bias
            if aid == 5:
                score += 0.03
            if aid in (1, 2, 3, 4):
                if self.last_changed_action_id == aid:
                    score += 0.08
                inv = self.INVERSE_ACTION.get(self.last_changed_action_id)
                if inv is not None and inv == aid:
                    score -= 0.10

        score += 1e-6 * self.py_rng.random()
        return float(score)

    def _segment_priority_key(self, seg: dict[str, Any]) -> tuple[Any, ...]:
        group = self.fp.segment_group(seg)
        color = int(seg["color"])
        salient = 0 if color in self.fp.salient_colors else 1
        status = 1 if color == self.fp.status_bar_color else 0
        area = int(seg["area"])
        x1, y1, x2, y2 = seg["bounding_box"]
        width = x2 - x1 + 1
        height = y2 - y1 + 1
        mediumish = 0 if (self.fp.minimal_width <= width <= self.fp.maximal_width and self.fp.minimal_width <= height <= self.fp.maximal_width) else 1
        return (group, status, salient, mediumish, -min(area, 256), y1, x1)

    def _build_view(self, obs: Any, env: Any) -> FrameView:
        raw_frame, num_frames = extract_latest_frame(obs)
        state = state_name(obs)
        lvl_done = levels_completed(obs)
        available_action_ids = extract_available_action_ids(obs, env)
        undo_available = 7 in available_action_ids

        if self.status_bar_mask is None or self.status_bar_mask.shape != raw_frame.shape:
            initial_labels, initial_segments = self.fp.segment_frame(raw_frame)
            self.status_bar_mask = self.fp.identify_status_bars(initial_labels, initial_segments)

        masked_for_seg = raw_frame.copy()
        masked_for_seg[self.status_bar_mask] = self.fp.status_bar_color
        hash_input = masked_for_seg.copy()
        hash_input[hash_input == self.fp.status_bar_color] = 0
        base_hash = self.fp.hash_frame(hash_input)
        action_sig = ",".join(str(a) for a in sorted(available_action_ids))
        node_id = f"{base_hash}|a:{action_sig}"

        if node_id in self.view_cache:
            cached = self.view_cache[node_id]
            return FrameView(
                node_id=cached.node_id,
                hash_id=cached.hash_id,
                raw_frame=raw_frame,
                masked_frame_for_segmentation=cached.masked_frame_for_segmentation,
                hash_frame_input=cached.hash_frame_input,
                num_frames=num_frames,
                state=state,
                levels_completed=lvl_done,
                available_action_ids=set(available_action_ids),
                undo_available=undo_available,
                segments=cached.segments,
                candidates=cached.candidates,
                groups=cached.groups,
            )

        labels, segments = self.fp.segment_frame(masked_for_seg)
        candidates: list[dict[str, Any]] = []
        candidate_groups: list[set[int]] = [set() for _ in range(self.n_groups)]

        if 6 in available_action_ids:
            seen_clicks: set[tuple[int, int]] = set()
            per_group_caps = [18, 14, 10, 8, 4]
            per_group_counts = [0 for _ in range(self.n_groups)]

            ordered_segments = sorted(enumerate(segments), key=lambda item: self._segment_priority_key(item[1]))
            for seg_id, seg in ordered_segments:
                base_group = self.fp.segment_group(seg)
                reps = self.fp.representative_points(seg)
                if int(seg["color"]) == self.fp.status_bar_color:
                    reps = reps[:1]
                    base_group = self.n_groups - 1

                for rep_idx, (x, y) in enumerate(reps):
                    group = min(base_group + (1 if rep_idx > 0 else 0), self.n_groups - 1)
                    if per_group_counts[group] >= per_group_caps[group]:
                        continue
                    key = (int(x), int(y))
                    if key in seen_clicks:
                        continue
                    seen_clicks.add(key)
                    cand_idx = len(candidates)
                    candidates.append(
                        {
                            "kind": "click",
                            "action_id": 6,
                            "data": {"x": int(x), "y": int(y)},
                            "group": group,
                            "tag": ("seg", seg_id, rep_idx, int(x), int(y)),
                            "features": self.fp.candidate_features(
                                kind="click",
                                action_id=6,
                                group=group,
                                x=int(x),
                                y=int(y),
                                raw_frame=raw_frame,
                                segment=seg,
                                source="segment",
                                rep_idx=rep_idx,
                            ),
                        }
                    )
                    candidate_groups[group].add(cand_idx)
                    per_group_counts[group] += 1

            invalid = self.status_bar_mask.copy()
            for x, y in seen_clicks:
                invalid[int(y), int(x)] = True
            click_only = set(available_action_ids).issubset({6, 7})
            grid_n = 5 if click_only else 4
            coarse = self.fp.coarse_grid_points(raw_frame, invalid_mask=invalid, grid_n=grid_n)
            coarse_cap = 10 if click_only else 6
            for j, (x, y) in enumerate(coarse[:coarse_cap]):
                group = self.n_groups - 2 if j < max(4, coarse_cap // 2) else self.n_groups - 1
                cand_idx = len(candidates)
                candidates.append(
                    {
                        "kind": "click",
                        "action_id": 6,
                        "data": {"x": int(x), "y": int(y)},
                        "group": group,
                        "tag": ("grid", j, int(x), int(y)),
                        "features": self.fp.candidate_features(
                            kind="click",
                            action_id=6,
                            group=group,
                            x=int(x),
                            y=int(y),
                            raw_frame=raw_frame,
                            segment=None,
                            source="grid",
                            rep_idx=0,
                        ),
                    }
                )
                candidate_groups[group].add(cand_idx)

        for aid in sorted(available_action_ids):
            if aid in {1, 2, 3, 4, 5}:
                cand_idx = len(candidates)
                candidates.append(
                    {
                        "kind": "simple",
                        "action_id": int(aid),
                        "data": None,
                        "group": 0,
                        "tag": ("simple", int(aid)),
                        "features": self.fp.candidate_features(
                            kind="simple",
                            action_id=int(aid),
                            group=0,
                            source="simple",
                        ),
                    }
                )
                candidate_groups[0].add(cand_idx)
            elif aid == 7:
                cand_idx = len(candidates)
                candidates.append(
                    {
                        "kind": "undo",
                        "action_id": 7,
                        "data": None,
                        "group": self.n_groups - 1,
                        "tag": ("undo", 7),
                        "features": {
                            "kind": "undo",
                            "action_id": 7,
                            "group": self.n_groups - 1,
                            "source": "undo",
                        },
                    }
                )
                candidate_groups[self.n_groups - 1].add(cand_idx)

        view = FrameView(
            node_id=node_id,
            hash_id=base_hash,
            raw_frame=raw_frame,
            masked_frame_for_segmentation=masked_for_seg,
            hash_frame_input=hash_input,
            num_frames=num_frames,
            state=state,
            levels_completed=lvl_done,
            available_action_ids=set(available_action_ids),
            undo_available=undo_available,
            segments=segments,
            candidates=candidates,
            groups=candidate_groups,
        )
        self.view_cache[node_id] = view
        return view

    def start_level(self, obs: Any, env: Any) -> FrameView:
        self.reset_level_state()
        view = self._build_view(obs, env)
        self.start_node = view.node_id
        self.graph.initialize(view.node_id, len(view.candidates), view.groups)
        self.last_unique_nodes = len(self.graph._nodes)
        return view

    def observe_transition(
        self,
        prev_view: FrameView,
        candidate_idx: int,
        next_obs: Any,
        env: Any,
        game_over: bool = False,
        level_up: bool = False,
    ) -> Optional[FrameView]:
        self.steps_in_level += 1
        candidate = prev_view.candidates[candidate_idx]

        if game_over:
            self.graph.record_test(prev_view.node_id, candidate_idx, 0, None)
            self.remember(candidate, changed=False, game_over=True)
            self.steps_since_hash_change += 1
            self.steps_since_new_node += 1
            return None

        if level_up:
            self.remember(candidate, changed=True, level_up=True)
            self.steps_since_hash_change = 0
            self.steps_since_new_node = 0
            aid = int(candidate.get("action_id", -1))
            if aid not in {7, -1}:
                self.last_changed_action_id = aid
            return None

        next_view = self._build_view(next_obs, env)
        changed = next_view.node_id != prev_view.node_id
        suspicious = next_view.node_id == self.start_node and next_view.num_frames > 1

        self.graph.record_test(
            prev_view.node_id,
            candidate_idx,
            1 if changed else 0,
            target_node=next_view.node_id if changed else None,
            target_num_candidates=len(next_view.candidates),
            group2remaining_candidate_ids=next_view.groups,
            suspicious_transition=suspicious,
        )
        self.remember(candidate, changed=changed, suspicious=suspicious)

        node_count = len(self.graph._nodes)
        if node_count > self.last_unique_nodes:
            self.steps_since_new_node = 0
            self.last_unique_nodes = node_count
        else:
            self.steps_since_new_node += 1

        if changed:
            self.steps_since_hash_change = 0
            aid = int(candidate.get("action_id", -1))
            if aid == 7 or candidate.get("kind") == "undo":
                if self.undo_stack:
                    if next_view.node_id == self.undo_stack[-1]:
                        self.undo_stack.pop()
                    elif next_view.node_id in self.undo_stack:
                        while self.undo_stack and self.undo_stack[-1] != next_view.node_id:
                            self.undo_stack.pop()
                        if self.undo_stack and self.undo_stack[-1] == next_view.node_id:
                            self.undo_stack.pop()
                self.last_changed_action_id = 7
            else:
                if prev_view.node_id != next_view.node_id:
                    if not self.undo_stack or self.undo_stack[-1] != prev_view.node_id:
                        self.undo_stack.append(prev_view.node_id)
                        if len(self.undo_stack) > 96:
                            self.undo_stack = self.undo_stack[-96:]
                self.last_changed_action_id = aid
        else:
            self.steps_since_hash_change += 1
        return next_view

    def note_external_reset(self, count: int = 1) -> None:
        self.resets_in_level += int(count)
        self.steps_in_level += int(count)
        self.steps_since_hash_change += int(count)
        self.steps_since_new_node += int(count)
        self.undo_stack.clear()
        self.last_changed_action_id = None

    def _undo_candidate_idx(self, view: FrameView) -> Optional[int]:
        for i, cand in enumerate(view.candidates):
            if int(cand.get("action_id", -1)) == 7:
                return i
        return None

    def choose_candidate(self, view: FrameView) -> Optional[dict[str, Any]]:
        if not view.candidates:
            return None

        undo_idx = self._undo_candidate_idx(view)
        undo_status = self.graph.edge_status(view.node_id, undo_idx) if undo_idx is not None else -1
        can_try_undo = (
            undo_idx is not None
            and bool(view.undo_available)
            and bool(self.undo_stack)
            and undo_status != -1
        )

        current_dist = self.graph.distance(view.node_id)
        unreachable = (
            view.node_id != self.start_node
            and view.node_id not in self.graph.frontier
            and current_dist >= INFINITY
        )

        level_idx = max(1, int(view.levels_completed) + 1)
        new_node_limit = 16 if level_idx <= 2 else (22 if level_idx <= 4 else 30)
        hash_change_limit = 10 if level_idx <= 2 else (14 if level_idx <= 4 else 20)
        stagnated = self.steps_since_new_node >= new_node_limit or self.steps_since_hash_change >= hash_change_limit

        if unreachable:
            if can_try_undo:
                out = dict(view.candidates[undo_idx])
                out["candidate_idx"] = int(undo_idx)
                out["reasoning"] = f"undo to recover reachable frontier from {view.node_id}"
                return out
            if view.node_id != self.start_node and self.resets_in_level < 12:
                return {"kind": "meta_reset", "reasoning": f"reset to recover frontier from {view.node_id}"}
            return None

        if stagnated and view.node_id != self.start_node:
            if can_try_undo:
                out = dict(view.candidates[undo_idx])
                out["candidate_idx"] = int(undo_idx)
                out["reasoning"] = f"undo to escape stagnation from {view.node_id}"
                return out
            if self.resets_in_level < 12:
                return {"kind": "meta_reset", "reasoning": f"reset after stagnation at {view.node_id}"}

        if view.node_id == self.start_node and not self.graph.frontier:
            return None

        open_ids = self.graph.open_candidate_ids(view.node_id)
        open_non_undo = [idx for idx in open_ids if int(view.candidates[idx].get("action_id", -1)) != 7]

        if open_non_undo:
            edge_idx = max(open_non_undo, key=lambda idx: self.candidate_priority(view.candidates[idx], path_mode=False))
            cand = view.candidates[edge_idx]
            out = dict(cand)
            out["candidate_idx"] = int(edge_idx)
            return out

        if can_try_undo:
            out = dict(view.candidates[undo_idx])
            out["candidate_idx"] = int(undo_idx)
            out["reasoning"] = f"undo to parent/frontier from {view.node_id}"
            return out

        path_ids = self.graph.successful_candidate_ids(view.node_id)
        path_non_undo = [idx for idx in path_ids if int(view.candidates[idx].get("action_id", -1)) != 7]

        if path_non_undo:
            edge_idx = max(
                path_non_undo,
                key=lambda idx: self.candidate_priority(
                    view.candidates[idx],
                    path_mode=True,
                    distance=self.graph.edge_distance(view.node_id, idx),
                ),
            )
            cand = view.candidates[edge_idx]
            out = dict(cand)
            out["candidate_idx"] = int(edge_idx)
            return out

        if view.node_id != self.start_node and self.resets_in_level < 12:
            return {"kind": "meta_reset", "reasoning": f"reset because no reachable frontier from {view.node_id}"}
        return None



try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import torch.optim as optim
    TORCH_AVAILABLE = True
except Exception:
    torch = None
    nn = None
    F = None
    optim = None
    TORCH_AVAILABLE = False

SIMPLE_MODEL_ACTION_IDS = [1, 2, 3, 4, 5, 7]
SIMPLE_MODEL_INDEX = {aid: idx for idx, aid in enumerate(SIMPLE_MODEL_ACTION_IDS)}


if TORCH_AVAILABLE:
    class ActionChangeModel(nn.Module):
        def __init__(self, in_channels: int = 17, width: int = 32, n_simple: int = len(SIMPLE_MODEL_ACTION_IDS)) -> None:
            super().__init__()
            w1 = int(width)
            w2 = int(width * 2)
            self.conv1 = nn.Conv2d(in_channels, w1, kernel_size=3, padding=1)
            self.conv2 = nn.Conv2d(w1, w2, kernel_size=3, padding=1)
            self.conv3 = nn.Conv2d(w2, w2, kernel_size=3, padding=1)
            self.simple_pool = nn.AdaptiveAvgPool2d((4, 4))
            self.simple_head = nn.Sequential(
                nn.Flatten(),
                nn.Linear(w2 * 4 * 4, 128),
                nn.ReLU(),
                nn.Linear(128, n_simple),
            )
            self.click_head = nn.Sequential(
                nn.Conv2d(w2, w1, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.Conv2d(w1, 1, kernel_size=1),
            )

        def forward(self, x: "torch.Tensor") -> tuple["torch.Tensor", "torch.Tensor"]:
            x = F.relu(self.conv1(x))
            x = F.relu(self.conv2(x))
            x = F.relu(self.conv3(x))
            simple_logits = self.simple_head(self.simple_pool(x))
            click_logits = self.click_head(x).flatten(1)
            return simple_logits, click_logits
else:
    ActionChangeModel = None


class HybridLevelExplorerAgent(LevelExplorerAgent):
    def __init__(self, game_id: str, n_groups: int = 5, global_memory: Optional[FeatureMemory] = None) -> None:
        self.device = None
        if TORCH_AVAILABLE:
            try:
                self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            except Exception:
                self.device = torch.device("cpu")
        self.model = None
        self.optimizer = None
        self.model_width = 40 if TORCH_AVAILABLE and self.device is not None and self.device.type == "cuda" else 24
        self.batch_size = 64 if TORCH_AVAILABLE and self.device is not None and self.device.type == "cuda" else 24
        self.train_frequency = 6 if TORCH_AVAILABLE and self.device is not None and self.device.type == "cuda" else 10
        self.min_train = max(24, self.batch_size)
        self.replay = deque(maxlen=24000)
        self.replay_seen: dict[str, int] = defaultdict(int)
        self.model_cache_node: Optional[str] = None
        self.model_cache_simple: Optional[np.ndarray] = None
        self.model_cache_click: Optional[np.ndarray] = None
        self.game_step_counter = 0
        self.node_candidate_failures: dict[tuple[str, int], int] = defaultdict(int)
        self.start_node_simple_untried: set[int] = set()
        self.start_node_click_untried: list[int] = []
        super().__init__(game_id, n_groups=n_groups, global_memory=global_memory)
        self._ensure_model()

    def _ensure_model(self) -> None:
        if not TORCH_AVAILABLE or ActionChangeModel is None:
            return
        if self.model is None:
            self.model = ActionChangeModel(width=self.model_width).to(self.device)
            self.optimizer = optim.Adam(self.model.parameters(), lr=3e-4)
            self.model.eval()

    def reset_level_state(self) -> None:
        super().reset_level_state()
        self.model_cache_node = None
        self.model_cache_simple = None
        self.model_cache_click = None
        self.node_candidate_failures = defaultdict(int)
        self.start_node_simple_untried = set()
        self.start_node_click_untried = []

    def _pad_frame(self, frame: np.ndarray) -> tuple[np.ndarray, tuple[int, int]]:
        frame = np.asarray(frame, dtype=np.uint8)
        h, w = frame.shape
        h = min(int(h), 64)
        w = min(int(w), 64)
        out = np.zeros((64, 64), dtype=np.uint8)
        out[:h, :w] = frame[:h, :w]
        return out, (h, w)

    def _frame_to_tensor_from_raw(self, frame: np.ndarray) -> Optional["torch.Tensor"]:
        if not TORCH_AVAILABLE or self.model is None:
            return None
        padded, (h, w) = self._pad_frame(frame)
        x = torch.zeros((17, 64, 64), dtype=torch.float32, device=self.device)
        frame_t = torch.from_numpy(padded.astype(np.int64)).to(self.device)
        x.scatter_(0, frame_t.unsqueeze(0), 1.0)
        x[16, :h, :w] = 1.0
        return x

    def _frame_to_tensor_from_exp(self, padded: np.ndarray, shape_hw: tuple[int, int]) -> Optional["torch.Tensor"]:
        if not TORCH_AVAILABLE or self.model is None:
            return None
        h, w = int(shape_hw[0]), int(shape_hw[1])
        h = max(0, min(64, h))
        w = max(0, min(64, w))
        x = torch.zeros((17, 64, 64), dtype=torch.float32, device=self.device)
        frame_t = torch.from_numpy(np.asarray(padded, dtype=np.int64)).to(self.device)
        x.scatter_(0, frame_t.unsqueeze(0), 1.0)
        if h > 0 and w > 0:
            x[16, :h, :w] = 1.0
        return x

    def _get_model_predictions(self, view: FrameView) -> tuple[Optional[np.ndarray], Optional[np.ndarray]]:
        if not TORCH_AVAILABLE or self.model is None:
            return None, None
        if self.model_cache_node == view.node_id and self.model_cache_simple is not None and self.model_cache_click is not None:
            return self.model_cache_simple, self.model_cache_click
        try:
            x = self._frame_to_tensor_from_raw(view.raw_frame)
            if x is None:
                return None, None
            self.model.eval()
            with torch.no_grad():
                simple_logits, click_logits = self.model(x.unsqueeze(0))
                simple_probs = torch.sigmoid(simple_logits[0]).detach().float().cpu().numpy()
                click_probs = torch.sigmoid(click_logits[0]).detach().float().cpu().numpy().reshape(64, 64)
            self.model_cache_node = view.node_id
            self.model_cache_simple = simple_probs
            self.model_cache_click = click_probs
            return simple_probs, click_probs
        except Exception:
            self.model_cache_node = None
            self.model_cache_simple = None
            self.model_cache_click = None
            return None, None

    def _candidate_model_score(self, view: FrameView, candidate: dict[str, Any]) -> float:
        simple_probs, click_probs = self._get_model_predictions(view)
        if simple_probs is None or click_probs is None:
            return 0.5
        aid = int(candidate.get("action_id", -1))
        if aid in SIMPLE_MODEL_INDEX:
            return float(simple_probs[SIMPLE_MODEL_INDEX[aid]])
        if aid == 6:
            data = candidate.get("data") or {}
            x = int(data.get("x", 0))
            y = int(data.get("y", 0))
            if 0 <= x < 64 and 0 <= y < 64:
                return float(click_probs[y, x])
        return 0.5

    def _candidate_to_exp_target(self, prev_view: FrameView, candidate: dict[str, Any], next_view: Optional[FrameView], *, game_over: bool, level_up: bool) -> float:
        if level_up:
            return 1.0
        if game_over:
            return 0.0
        if next_view is None:
            return 0.0
        changed = next_view.node_id != prev_view.node_id
        suspicious = next_view.node_id == self.start_node and next_view.num_frames > 1
        if changed and not suspicious:
            return 0.80
        if changed and suspicious:
            return 0.35
        return 0.0

    def _candidate_to_exp_index(self, candidate: dict[str, Any]) -> Optional[tuple[str, int]]:
        aid = int(candidate.get("action_id", -1))
        if aid in SIMPLE_MODEL_INDEX:
            return ("simple", int(SIMPLE_MODEL_INDEX[aid]))
        if aid == 6:
            data = candidate.get("data") or {}
            x = int(data.get("x", 0))
            y = int(data.get("y", 0))
            if 0 <= x < 64 and 0 <= y < 64:
                return ("click", int(y * 64 + x))
        return None

    def _record_experience(self, prev_view: FrameView, candidate: dict[str, Any], next_view: Optional[FrameView], *, game_over: bool, level_up: bool) -> None:
        if not TORCH_AVAILABLE or self.model is None:
            return
        idx_info = self._candidate_to_exp_index(candidate)
        if idx_info is None:
            return
        kind, idx = idx_info
        target = float(self._candidate_to_exp_target(prev_view, candidate, next_view, game_over=game_over, level_up=level_up))
        padded, shape_hw = self._pad_frame(prev_view.raw_frame)
        key_payload = padded.tobytes() + str(shape_hw).encode("utf-8") + f"|{kind}|{idx}".encode("utf-8")
        key = hashlib.blake2b(key_payload, digest_size=16).hexdigest()
        cap = 2 if target <= 0.0 else 3
        if self.replay_seen.get(key, 0) >= cap:
            return
        self.replay.append(
            {
                "frame": padded.copy(),
                "shape_hw": tuple(shape_hw),
                "kind": kind,
                "idx": int(idx),
                "target": float(target),
            }
        )
        self.replay_seen[key] += 1

    def _train_model(self, train_steps: int = 1) -> None:
        if not TORCH_AVAILABLE or self.model is None or self.optimizer is None:
            return
        if len(self.replay) < self.min_train:
            return
        try:
            self.model.train()
            replay_size = len(self.replay)
            batch_n = min(self.batch_size, replay_size)
            for _ in range(max(1, int(train_steps))):
                batch_indices = self.np_rng.choice(replay_size, size=batch_n, replace=False)
                batch = [self.replay[int(i)] for i in batch_indices]
                xs = []
                for exp in batch:
                    x = self._frame_to_tensor_from_exp(exp["frame"], exp["shape_hw"])
                    if x is not None:
                        xs.append(x)
                if len(xs) != len(batch) or not xs:
                    continue
                x_batch = torch.stack(xs, dim=0)
                y_batch = torch.tensor([float(exp["target"]) for exp in batch], dtype=torch.float32, device=self.device)
                simple_logits, click_logits = self.model(x_batch)

                selected_logits = []
                for i, exp in enumerate(batch):
                    if exp["kind"] == "simple":
                        selected_logits.append(simple_logits[i, int(exp["idx"])])
                    else:
                        selected_logits.append(click_logits[i, int(exp["idx"])])
                selected_logits = torch.stack(selected_logits, dim=0)

                loss = F.binary_cross_entropy_with_logits(selected_logits, y_batch)
                # Mild exploration regularization to avoid total collapse early in the game.
                loss = loss - 2e-4 * torch.sigmoid(simple_logits).mean() - 2e-5 * torch.sigmoid(click_logits).mean()

                self.optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.optimizer.step()
            self.model.eval()
            self.model_cache_node = None
            self.model_cache_simple = None
            self.model_cache_click = None
        except Exception:
            try:
                self.model.eval()
            except Exception:
                pass

    def start_level(self, obs: Any, env: Any) -> FrameView:
        view = super().start_level(obs, env)
        self.start_node_simple_untried = {
            idx for idx, cand in enumerate(view.candidates)
            if cand.get("kind") == "simple" and int(cand.get("action_id", -1)) != 7
        }
        self.start_node_click_untried = [
            idx for idx, cand in enumerate(view.candidates)
            if cand.get("kind") == "click" and int(cand.get("group", self.n_groups - 1)) <= 1
        ][:8]
        return view

    def note_external_reset(self, count: int = 1) -> None:
        super().note_external_reset(count)
        self.model_cache_node = None
        self.model_cache_simple = None
        self.model_cache_click = None

    def _node_candidate_key(self, view: FrameView, candidate_idx: int) -> tuple[str, int]:
        return (str(view.node_id), int(candidate_idx))

    def hybrid_score(self, view: FrameView, candidate: dict[str, Any], *, path_mode: bool = False, distance: int = 0) -> float:
        base = super().candidate_priority(candidate, path_mode=path_mode, distance=distance)
        base_prob = 1.0 / (1.0 + math.exp(-1.45 * float(base)))
        model_prob = self._candidate_model_score(view, candidate)
        local_prior = self.level_memory.estimate(candidate)
        global_prior = self.global_memory.estimate(candidate)
        prior = 0.72 * local_prior + 0.28 * global_prior

        node_failures = self.node_candidate_failures.get(self._node_candidate_key(view, int(candidate.get("candidate_idx", -1))), 0)
        fail_penalty = 0.06 * min(4, int(node_failures))

        kind = str(candidate.get("kind", ""))
        aid = int(candidate.get("action_id", -1))
        warmup_bonus = 0.0
        if not path_mode and view.node_id == self.start_node:
            if kind == "simple" and aid != 7 and self.steps_in_level <= max(4, len(view.available_action_ids)):
                warmup_bonus += 0.08
            elif kind == "click" and self.steps_in_level <= 8 and int(candidate.get("group", self.n_groups - 1)) <= 1:
                warmup_bonus += 0.04

        if path_mode:
            score = 0.54 * base_prob + 0.14 * model_prob + 0.32 * prior - 0.018 * float(distance) - fail_penalty
        else:
            score = 0.37 * base_prob + 0.38 * model_prob + 0.25 * prior + warmup_bonus - fail_penalty

        score += 1e-6 * self.py_rng.random()
        return float(score)

    def observe_transition(
        self,
        prev_view: FrameView,
        candidate_idx: int,
        next_obs: Any,
        env: Any,
        game_over: bool = False,
        level_up: bool = False,
    ) -> Optional[FrameView]:
        candidate = prev_view.candidates[candidate_idx]
        next_view = super().observe_transition(
            prev_view,
            candidate_idx,
            next_obs,
            env,
            game_over=game_over,
            level_up=level_up,
        )
        try:
            self._record_experience(prev_view, candidate, next_view, game_over=game_over, level_up=level_up)
        except Exception:
            pass

        key = self._node_candidate_key(prev_view, candidate_idx)
        changed = bool(level_up) or (next_view is not None and next_view.node_id != prev_view.node_id)
        if changed:
            self.node_candidate_failures[key] = max(0, int(self.node_candidate_failures.get(key, 0)) - 1)
        else:
            self.node_candidate_failures[key] += 1

        self.game_step_counter += 1
        if TORCH_AVAILABLE and self.model is not None and self.game_step_counter % self.train_frequency == 0:
            extra = 2 if self.device is not None and self.device.type == "cuda" and len(self.replay) >= 4 * self.batch_size else 1
            self._train_model(train_steps=extra)
        return next_view

    def choose_candidate(self, view: FrameView) -> Optional[dict[str, Any]]:
        if not view.candidates:
            return None

        # Cache predictions once for the current view so repeated scoring is cheap.
        self._get_model_predictions(view)

        undo_idx = self._undo_candidate_idx(view)
        undo_status = self.graph.edge_status(view.node_id, undo_idx) if undo_idx is not None else -1
        can_try_undo = (
            undo_idx is not None
            and bool(view.undo_available)
            and bool(self.undo_stack)
            and undo_status != -1
        )

        current_dist = self.graph.distance(view.node_id)
        unreachable = (
            view.node_id != self.start_node
            and view.node_id not in self.graph.frontier
            and current_dist >= INFINITY
        )

        level_idx = max(1, int(view.levels_completed) + 1)
        max_resets = 8 if level_idx <= 3 else 10
        new_node_limit = 14 if level_idx <= 2 else (20 if level_idx <= 4 else 28)
        hash_change_limit = 9 if level_idx <= 2 else (13 if level_idx <= 4 else 18)
        stagnated = self.steps_since_new_node >= new_node_limit or self.steps_since_hash_change >= hash_change_limit

        if unreachable:
            if can_try_undo:
                out = dict(view.candidates[undo_idx])
                out["candidate_idx"] = int(undo_idx)
                out["reasoning"] = f"undo to recover reachable frontier from {view.node_id}"
                return out
            if view.node_id != self.start_node and self.resets_in_level < max_resets:
                return {"kind": "meta_reset", "reasoning": f"reset to recover frontier from {view.node_id}"}
            return None

        if stagnated and view.node_id != self.start_node:
            if can_try_undo:
                out = dict(view.candidates[undo_idx])
                out["candidate_idx"] = int(undo_idx)
                out["reasoning"] = f"undo to escape stagnation from {view.node_id}"
                return out
            if self.resets_in_level < max_resets:
                return {"kind": "meta_reset", "reasoning": f"reset after stagnation at {view.node_id}"}

        if view.node_id == self.start_node and not self.graph.frontier:
            return None

        open_ids = self.graph.open_candidate_ids(view.node_id)
        open_non_undo = [idx for idx in open_ids if int(view.candidates[idx].get("action_id", -1)) != 7]

        # Warm start on the first node: probe simple actions before aggressive clicks.
        if view.node_id == self.start_node and open_non_undo:
            simple_candidates = [idx for idx in open_non_undo if idx in self.start_node_simple_untried]
            if simple_candidates and self.steps_in_level <= max(4, len(view.available_action_ids)):
                edge_idx = max(
                    simple_candidates,
                    key=lambda idx: self.hybrid_score(view, {**view.candidates[idx], "candidate_idx": int(idx)}, path_mode=False),
                )
                self.start_node_simple_untried.discard(edge_idx)
                cand = view.candidates[edge_idx]
                out = dict(cand)
                out["candidate_idx"] = int(edge_idx)
                return out
            if self.start_node_click_untried and self.steps_in_level <= 8:
                click_candidates = [idx for idx in self.start_node_click_untried if idx in open_non_undo]
                if click_candidates:
                    edge_idx = max(
                        click_candidates,
                        key=lambda idx: self.hybrid_score(view, {**view.candidates[idx], "candidate_idx": int(idx)}, path_mode=False),
                    )
                    self.start_node_click_untried = [idx for idx in self.start_node_click_untried if idx != edge_idx]
                    cand = view.candidates[edge_idx]
                    out = dict(cand)
                    out["candidate_idx"] = int(edge_idx)
                    return out

        if open_non_undo:
            edge_idx = max(
                open_non_undo,
                key=lambda idx: self.hybrid_score(view, {**view.candidates[idx], "candidate_idx": int(idx)}, path_mode=False),
            )
            cand = view.candidates[edge_idx]
            out = dict(cand)
            out["candidate_idx"] = int(edge_idx)
            return out

        if can_try_undo:
            out = dict(view.candidates[undo_idx])
            out["candidate_idx"] = int(undo_idx)
            out["reasoning"] = f"undo to parent/frontier from {view.node_id}"
            return out

        path_ids = self.graph.successful_candidate_ids(view.node_id)
        path_non_undo = [idx for idx in path_ids if int(view.candidates[idx].get("action_id", -1)) != 7]

        if path_non_undo:
            edge_idx = max(
                path_non_undo,
                key=lambda idx: self.hybrid_score(
                    view,
                    {**view.candidates[idx], "candidate_idx": int(idx)},
                    path_mode=True,
                    distance=self.graph.edge_distance(view.node_id, idx),
                ),
            )
            cand = view.candidates[edge_idx]
            out = dict(cand)
            out["candidate_idx"] = int(edge_idx)
            return out

        if view.node_id != self.start_node and self.resets_in_level < max_resets:
            return {"kind": "meta_reset", "reasoning": f"reset because no reachable frontier from {view.node_id}"}
        return None

def level_budget(level_idx: int) -> int:
    table = {
        1: 80,
        2: 120,
        3: 180,
        4: 260,
        5: 360,
        6: 480,
        7: 640,
    }
    return table.get(level_idx, 760)


def game_budget(level_idx: int) -> int:
    return 1100 + 200 * max(0, level_idx - 1)


def make_env(arc: Any, game_id: str) -> Any:
    sig = inspect.signature(arc.make)
    kwargs: dict[str, Any] = {}
    if "save_recording" in sig.parameters:
        kwargs["save_recording"] = False
    if "include_frame_data" in sig.parameters:
        kwargs["include_frame_data"] = True
    if "render_mode" in sig.parameters:
        kwargs["render_mode"] = None
    return arc.make(game_id, **kwargs)




# ==================== Local source planner ====================

try:
    ActionInput = find_symbol("arcengine", "ActionInput")
except Exception:
    try:
        ActionInput = find_symbol("arc_agi", "ActionInput")
    except Exception:
        ActionInput = None


class _FallbackActionInput:
    def __init__(self, id: Any = None, action: Any = None, data: Optional[dict[str, Any]] = None, **kwargs: Any) -> None:
        self.id = id if id is not None else action
        self.action = action if action is not None else id
        self.data = data
        for k, v in kwargs.items():
            setattr(self, k, v)


PLANNER_LOG = logging.getLogger("arc_source_planner")
if not PLANNER_LOG.handlers:
    logging.basicConfig(level=logging.WARNING)
PLANNER_LOG.setLevel(logging.INFO if os.getenv("ARC_DEBUG", "0") == "1" else logging.WARNING)

ENV_SOURCE_CACHE: dict[str, tuple[Optional[str], Optional[str]]] = {}
GAME_CLASS_CACHE: dict[tuple[str, str], Any] = {}
ENV_FILE_INDEX: dict[str, list[Path]] = defaultdict(list)
if ENVIRONMENTS_DIR.exists():
    for _py_path in ENVIRONMENTS_DIR.rglob("*.py"):
        if _py_path.name.startswith("_"):
            continue
        ENV_FILE_INDEX[_py_path.stem.lower()].append(_py_path)

NOISY_FIELD_TOKENS = {
    "logger", "logging", "recorder", "record", "recording",
    "screen", "surface", "image", "images", "sprite", "sprites",
    "render", "renderer", "display", "window", "sound", "audio",
    "clock", "timer", "rng", "random", "seed", "api", "client",
    "socket", "session", "history", "trace", "debug", "cache",
    "memo", "frames", "frame", "pixels", "grid_cache", "tile_cache",
    "guid", "uuid", "id_map", "path_cache", "log", "profile",
}
NOISY_FIELD_EXACT = {
    "_action_count", "action_count", "_last_action", "last_action",
    "_action_complete", "_full_reset", "_elapsed", "_elapsed_steps",
    "_frame_index", "_tick", "tick", "_guid",
}
SEMANTIC_PATH_TOKENS = {
    "goal": 8,
    "target": 8,
    "exit": 8,
    "finish": 8,
    "flag": 7,
    "home": 7,
    "portal": 7,
    "key": 6,
    "door": 6,
    "switch": 6,
    "button": 6,
    "cursor": 6,
    "player": 6,
    "agent": 6,
    "avatar": 6,
    "hero": 6,
    "enemy": 5,
    "monster": 5,
    "object": 4,
    "box": 4,
    "gem": 4,
    "coin": 4,
    "selected": 5,
    "holding": 5,
    "held": 5,
    "mode": 5,
    "phase": 5,
    "state": 4,
    "index": 2,
    "count": 2,
    "pos": 3,
    "coord": 3,
    "x": 1,
    "y": 1,
    "row": 1,
    "col": 1,
    "cx": 1,
    "cy": 1,
}


def planner_time_budget_seconds(level_idx: int, has_transfer: bool = False) -> float:
    table = {
        0: 12.0,
        1: 18.0,
        2: 28.0,
        3: 42.0,
        4: 58.0,
        5: 78.0,
        6: 98.0,
    }
    budget = table.get(int(level_idx), 110.0)
    if has_transfer:
        budget *= 1.15
    return float(budget)


def _path_text(path: tuple[Any, ...]) -> str:
    return ".".join(str(p).lower() for p in path)


def _semantic_path_score(path: tuple[Any, ...]) -> int:
    text = _path_text(path)
    score = 0
    for token, bonus in SEMANTIC_PATH_TOKENS.items():
        if token in text:
            score += int(bonus)
    return int(score)


def _is_noisy_field(name: Any) -> bool:
    s = str(name).strip().lower()
    if s in NOISY_FIELD_EXACT:
        return True
    if s.startswith("__") and s.endswith("__"):
        return True
    for tok in NOISY_FIELD_TOKENS:
        if tok in s:
            return True
    return False


def _normalize_number(v: Any) -> Any:
    if isinstance(v, np.generic):
        v = v.item()
    if isinstance(v, bool):
        return bool(v)
    if isinstance(v, (int, np.integer)):
        return int(v)
    if isinstance(v, (float, np.floating)):
        if math.isnan(float(v)) or math.isinf(float(v)):
            return None
        return round(float(v), 4)
    return v


def _pack_small_value(v: Any) -> Any:
    if v is None:
        return None
    if isinstance(v, np.generic):
        v = v.item()
    if isinstance(v, (bool, int, np.integer, float, np.floating)):
        return _normalize_number(v)
    if isinstance(v, str):
        if len(v) <= 48:
            return v
        return v[:48]
    if isinstance(v, bytes):
        if len(v) <= 16:
            return tuple(int(x) for x in v)
        return None
    if isinstance(v, np.ndarray):
        if v.size <= 16:
            arr = np.asarray(v)
            flat = tuple(_normalize_number(x) for x in arr.flatten().tolist())
            return ("nd", tuple(int(x) for x in arr.shape), flat)
        return None
    if isinstance(v, (list, tuple)):
        if len(v) <= 6:
            packed = []
            for item in v:
                pv = _pack_small_value(item)
                if pv is None:
                    return None
                packed.append(pv)
            return tuple(packed)
        return None
    if isinstance(v, dict):
        if len(v) <= 8:
            packed_items = []
            for key in sorted(v.keys(), key=lambda z: str(z))[:8]:
                pv = _pack_small_value(v[key])
                if pv is None:
                    continue
                packed_items.append((str(key), pv))
            if packed_items:
                return tuple(packed_items)
        return None
    if hasattr(v, "name") and isinstance(getattr(v, "name"), str):
        return str(v.name)
    return None


def _flatten_small_state(obj: Any, depth: int = 2, max_items: int = 160) -> dict[tuple[Any, ...], Any]:
    out: dict[tuple[Any, ...], Any] = {}
    seen: set[int] = set()

    def rec(x: Any, path: tuple[Any, ...], d: int) -> None:
        if d < 0 or x is None or len(out) >= max_items:
            return
        try:
            oid = id(x)
            if oid in seen:
                return
            seen.add(oid)
        except Exception:
            pass

        packed = _pack_small_value(x)
        if packed is not None:
            out[path] = packed
            return

        if isinstance(x, dict):
            items = list(x.items())[:12]
            for k, v in items:
                if _is_noisy_field(k):
                    continue
                rec(v, path + (str(k),), d - 1)
            return

        if isinstance(x, (list, tuple)):
            if len(x) <= 8:
                for i, v in enumerate(x[:8]):
                    rec(v, path + (int(i),), d - 1)
            return

        attrs = getattr(x, "__dict__", None)
        if isinstance(attrs, dict):
            items = sorted(attrs.items(), key=lambda kv: str(kv[0]))[:24]
            for k, v in items:
                if _is_noisy_field(k):
                    continue
                rec(v, path + (str(k),), d - 1)

    rec(obj, tuple(), depth)
    return out


def _get_value_by_path(obj: Any, path: tuple[Any, ...]) -> Any:
    x = obj
    for key in path:
        if x is None:
            return None
        try:
            if isinstance(key, int):
                if isinstance(x, (list, tuple)) and 0 <= key < len(x):
                    x = x[key]
                else:
                    return None
            else:
                if isinstance(x, dict):
                    if key in x:
                        x = x[key]
                    elif str(key) in x:
                        x = x[str(key)]
                    else:
                        return None
                else:
                    if not hasattr(x, key):
                        return None
                    x = getattr(x, key)
        except Exception:
            return None
    return _pack_small_value(x)


def _hidden_coord_points_from_obj(obj: Any, shape_hw: tuple[int, int], max_points: int = 24) -> list[tuple[float, int, int, str]]:
    h, w = int(shape_hw[0]), int(shape_hw[1])
    out: list[tuple[float, int, int, str]] = []
    seen_pts: set[tuple[int, int]] = set()
    seen_objs: set[int] = set()

    def add_pt(x: Any, y: Any, score: float, source: str) -> None:
        try:
            xi = int(round(float(_normalize_number(x))))
            yi = int(round(float(_normalize_number(y))))
        except Exception:
            return
        if not (0 <= xi < w and 0 <= yi < h):
            return
        key = (xi, yi)
        if key in seen_pts:
            return
        seen_pts.add(key)
        out.append((float(score), xi, yi, source))

    def rec(x: Any, path: tuple[Any, ...], depth: int) -> None:
        if depth < 0 or x is None or len(out) >= max_points:
            return
        try:
            oid = id(x)
            if oid in seen_objs:
                return
            seen_objs.add(oid)
        except Exception:
            pass

        path_score = _semantic_path_score(path)
        text = _path_text(path)

        if isinstance(x, dict):
            lower_map = {str(k).lower(): k for k in x.keys()}
            if "x" in lower_map and "y" in lower_map:
                add_pt(x[lower_map["x"]], x[lower_map["y"]], 80 + path_score, f"hidden:{text or 'xy'}")
            if "col" in lower_map and "row" in lower_map:
                add_pt(x[lower_map["col"]], x[lower_map["row"]], 78 + path_score, f"hidden:{text or 'rc'}")
            if "cx" in lower_map and "cy" in lower_map:
                add_pt(x[lower_map["cx"]], x[lower_map["cy"]], 76 + path_score, f"hidden:{text or 'cxy'}")
            items = list(x.items())[:12]
            for k, v in items:
                if _is_noisy_field(k):
                    continue
                rec(v, path + (str(k),), depth - 1)
            return

        if isinstance(x, (list, tuple)):
            if len(x) >= 2 and all(isinstance(v, (int, float, np.integer, np.floating, bool)) for v in x[:2]):
                a, b = x[0], x[1]
                add_pt(a, b, 64 + path_score, f"hidden:{text or 'pair'}")
                add_pt(b, a, 60 + path_score, f"hidden:{text or 'pair_rev'}")
            if len(x) <= 8:
                for i, v in enumerate(x[:8]):
                    rec(v, path + (int(i),), depth - 1)
            return

        attrs = getattr(x, "__dict__", None)
        if isinstance(attrs, dict):
            lower_map = {str(k).lower(): k for k in attrs.keys()}
            if "x" in lower_map and "y" in lower_map:
                add_pt(getattr(x, lower_map["x"]), getattr(x, lower_map["y"]), 80 + path_score, f"hidden:{text or 'xy'}")
            if "col" in lower_map and "row" in lower_map:
                add_pt(getattr(x, lower_map["col"]), getattr(x, lower_map["row"]), 78 + path_score, f"hidden:{text or 'rc'}")
            if "cx" in lower_map and "cy" in lower_map:
                add_pt(getattr(x, lower_map["cx"]), getattr(x, lower_map["cy"]), 76 + path_score, f"hidden:{text or 'cxy'}")
            if "position" in lower_map:
                rec(getattr(x, lower_map["position"]), path + ("position",), depth - 1)
            items = sorted(attrs.items(), key=lambda kv: str(kv[0]))[:24]
            for k, v in items:
                if _is_noisy_field(k):
                    continue
                rec(v, path + (str(k),), depth - 1)

    rec(obj, tuple(), 3)
    out.sort(key=lambda t: (-t[0], t[2], t[1]))
    return out[:max_points]


def _visible_objects(frame: np.ndarray) -> list[dict[str, Any]]:
    frame = np.asarray(frame, dtype=np.uint8)
    bg = int(np.bincount(frame.reshape(-1), minlength=16).argmax())
    objs = []
    for color in range(16):
        if color == bg:
            continue
        mask = frame == color
        count = int(mask.sum())
        if count < 2:
            continue
        ys, xs = np.where(mask)
        objs.append(
            {
                "color": int(color),
                "cx": float(xs.mean()),
                "cy": float(ys.mean()),
                "n": int(count),
            }
        )
    objs.sort(key=lambda o: (o["color"], -o["n"], o["cx"], o["cy"]))
    return objs


def _estimate_click_offset(prev_frame: np.ndarray, curr_frame: np.ndarray) -> tuple[float, float]:
    objs_prev = _visible_objects(prev_frame)
    objs_curr = _visible_objects(curr_frame)
    if not objs_prev or not objs_curr:
        return 0.0, 0.0
    matched = []
    for op in objs_prev:
        best = None
        best_dist = float("inf")
        for oc in objs_curr:
            if oc["color"] != op["color"]:
                continue
            if abs(oc["n"] - op["n"]) > max(op["n"], oc["n"]) * 0.55:
                continue
            d = abs(oc["cx"] - op["cx"]) + abs(oc["cy"] - op["cy"])
            if d < best_dist:
                best_dist = d
                best = oc
        if best is not None:
            matched.append((op, best))
    if not matched:
        return 0.0, 0.0
    dx = float(np.mean([m[1]["cx"] - m[0]["cx"] for m in matched]))
    dy = float(np.mean([m[1]["cy"] - m[0]["cy"] for m in matched]))
    if abs(dx) > curr_frame.shape[1] or abs(dy) > curr_frame.shape[0]:
        return 0.0, 0.0
    return dx, dy


def _find_game_source_and_class(game_id: str, env: Any = None) -> tuple[Optional[str], Optional[str]]:
    if game_id in ENV_SOURCE_CACHE:
        return ENV_SOURCE_CACHE[game_id]

    base = str(game_id).split("-")[0]
    suffix = str(game_id).split("-", 1)[1] if "-" in str(game_id) else ""
    candidates: list[tuple[int, Path]] = []

    def add_candidate(path_like: Any, bonus: int = 0) -> None:
        if path_like is None:
            return
        path = Path(path_like)
        if not path.exists():
            return
        score = int(bonus)
        path_text = str(path).lower()
        if path.stem.lower() == base.lower():
            score += 30
        if suffix and suffix.lower() in path_text:
            score += 18
        if base.lower() in path_text:
            score += 8
        score += max(0, 6 - len(path.parts))
        candidates.append((score, path))

    local_dir = None
    try:
        env_info = maybe_get(env, "environment_info")
        local_dir = maybe_get(env_info, "local_dir")
    except Exception:
        local_dir = None
    if local_dir:
        local_dir = Path(local_dir)
        add_candidate(local_dir / f"{base}.py", bonus=40)
        for py in local_dir.glob("*.py"):
            add_candidate(py, bonus=25)

    exact_candidates = [
        ENVIRONMENTS_DIR / base / suffix / f"{base}.py",
        ENVIRONMENTS_DIR / base / f"{base}.py",
        ENVIRONMENTS_DIR / f"{base}.py",
    ]
    for cand in exact_candidates:
        add_candidate(cand, bonus=20)

    for py in ENV_FILE_INDEX.get(base.lower(), []):
        add_candidate(py, bonus=10)

    if not candidates:
        ENV_SOURCE_CACHE[game_id] = (None, None)
        return (None, None)

    candidates.sort(key=lambda t: (-t[0], len(str(t[1]))))
    chosen = candidates[0][1]
    class_name = None
    try:
        text = chosen.read_text(encoding="utf-8", errors="ignore")[:20000]
        m = re.search(r"class\s+([A-Za-z_][A-Za-z0-9_]*)\s*\(\s*ARCBaseGame", text)
        if m:
            class_name = m.group(1)
        else:
            class_candidates = re.findall(r"class\s+([A-Za-z_][A-Za-z0-9_]*)\s*\(", text)
            if class_candidates:
                for cand in class_candidates:
                    if cand.lower().startswith(base.lower()):
                        class_name = cand
                        break
                if class_name is None:
                    class_name = class_candidates[0]
    except Exception:
        class_name = None

    if class_name is None:
        class_name = base[:1].upper() + base[1:]

    out = (str(chosen), str(class_name))
    ENV_SOURCE_CACHE[game_id] = out
    return out


def _load_game_class(source_path: str, class_name: str) -> Any:
    key = (str(source_path), str(class_name))
    if key in GAME_CLASS_CACHE:
        return GAME_CLASS_CACHE[key]

    module_name = f"_arc_local_{hashlib.md5(str(source_path).encode()).hexdigest()[:12]}"
    parent = str(Path(source_path).parent)
    grandparent = str(Path(source_path).parent.parent)
    if parent not in sys.path:
        sys.path.insert(0, parent)
    if grandparent not in sys.path:
        sys.path.insert(0, grandparent)
    if str(ENVIRONMENTS_DIR) not in sys.path:
        sys.path.insert(0, str(ENVIRONMENTS_DIR))

    spec = importlib.util.spec_from_file_location(module_name, source_path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load spec for {source_path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)

    if hasattr(module, class_name):
        cls = getattr(module, class_name)
    else:
        cls = None
        for name, obj in vars(module).items():
            if inspect.isclass(obj) and name.lower().startswith(Path(source_path).stem.lower()):
                cls = obj
                break
        if cls is None:
            for name, obj in vars(module).items():
                if inspect.isclass(obj) and hasattr(obj, "perform_action"):
                    cls = obj
                    break
    if cls is None:
        raise AttributeError(f"Could not resolve game class '{class_name}' from {source_path}")

    GAME_CLASS_CACHE[key] = cls
    return cls


def _game_action_token(action_id: int) -> Any:
    try:
        if hasattr(GameAction, "from_id"):
            return GameAction.from_id(int(action_id))
    except Exception:
        pass
    if int(action_id) == 0:
        return RESET_ACTION
    return ACTION_ID_TO_ENUM.get(int(action_id), int(action_id))


def _make_local_action_input(action_id: int, data: Optional[dict[str, Any]] = None) -> Any:
    payload = None
    if data is not None:
        payload = {}
        for k, v in data.items():
            try:
                payload[str(k)] = int(v) if isinstance(v, (int, np.integer, bool)) else float(v) if isinstance(v, (float, np.floating)) else v
            except Exception:
                payload[str(k)] = v
    token = _game_action_token(action_id)
    cls = ActionInput or _FallbackActionInput

    attempts: list[tuple[tuple[Any, ...], dict[str, Any]]] = []
    if payload and "x" in payload and "y" in payload:
        attempts.extend(
            [
                (tuple(), {"id": token, "x": payload["x"], "y": payload["y"], "data": payload}),
                (tuple(), {"action": token, "x": payload["x"], "y": payload["y"], "data": payload}),
            ]
        )
    attempts.extend(
        [
            (tuple(), {"id": token, "data": payload}),
            (tuple(), {"action": token, "data": payload}),
            ((token, payload), {}),
            ((token,), {"data": payload}),
            ((token,), {}),
        ]
    )

    for args, kwargs in attempts:
        try:
            return cls(*args, **{k: v for k, v in kwargs.items() if v is not None})
        except Exception:
            continue
    return _FallbackActionInput(id=token, data=payload)


def _local_step(game: Any, action_id: int, data: Optional[dict[str, Any]] = None) -> Any:
    fn = getattr(game, "perform_action", None)
    if fn is None:
        fn = getattr(game, "step", None)
    if fn is None:
        raise AttributeError("Local game object has neither perform_action nor step.")

    token = _game_action_token(action_id)
    ai = _make_local_action_input(action_id, data=data)

    patterns: list[tuple[tuple[Any, ...], dict[str, Any]]] = []
    patterns.extend(
        [
            ((ai,), {"raw": True}),
            ((ai,), {}),
        ]
    )
    if data is not None:
        patterns.extend(
            [
                ((token,), {"data": data, "raw": True}),
                ((token,), {"data": data}),
                ((int(action_id),), {"data": data, "raw": True}),
                ((int(action_id),), {"data": data}),
                ((token, data), {}),
                ((int(action_id), data), {}),
            ]
        )
    patterns.extend(
        [
            ((token,), {"raw": True}),
            ((token,), {}),
            ((int(action_id),), {"raw": True}),
            ((int(action_id),), {}),
        ]
    )

    last_exc = None
    for args, kwargs in patterns:
        try:
            return fn(*args, **{k: v for k, v in kwargs.items() if v is not None})
        except TypeError as exc:
            last_exc = exc
            continue
        except Exception as exc:
            last_exc = exc
            continue
    if last_exc is None:
        raise RuntimeError(f"Could not apply local action {action_id}")
    raise last_exc


def _local_frame_from_result(result: Any, game: Any) -> Optional[np.ndarray]:
    frame_obj = maybe_get(result, "frame", "frames", "grid", "grids")
    if frame_obj is None:
        frame_obj = deep_find(result, ("frame", "frames", "grid", "grids"), 3)
    if frame_obj is not None:
        try:
            arr = np.asarray(frame_obj, dtype=np.uint8)
            if arr.ndim == 3:
                return arr[-1].copy()
            if arr.ndim == 2:
                return arr.copy()
        except Exception:
            pass
    if hasattr(game, "get_pixels"):
        try:
            arr = np.asarray(game.get_pixels(0, 0, 64, 64), dtype=np.uint8)
            if arr.ndim == 3:
                return arr[-1].copy()
            if arr.ndim == 2:
                return arr.copy()
        except Exception:
            return None
    return None


def _local_state_name(result: Any, game: Any) -> str:
    st = maybe_get(result, "state", "game_state")
    if st is None:
        st = deep_find(result, ("state", "game_state"), 2)
    if st is None:
        st = maybe_get(game, "_game_state", "game_state", "state")
    if st is None:
        return ""
    if isinstance(st, str):
        return st.split(".")[-1].upper()
    if hasattr(st, "name"):
        return str(st.name).upper()
    return str(st).split(".")[-1].upper()


def _local_levels_completed(result: Any, game: Any) -> int:
    val = maybe_get(result, "levels_completed", "score")
    if val is None:
        val = deep_find(result, ("levels_completed", "score"), 2)
    if val is None:
        val = maybe_get(game, "_score", "score")
    if val is None:
        return 0
    try:
        return int(val)
    except Exception:
        try:
            return int(float(val))
        except Exception:
            return 0


def _local_current_level_index(game: Any, result: Any = None) -> Optional[int]:
    val = maybe_get(game, "_current_level_index", "current_level_index", "level_index")
    if val is None and result is not None:
        val = maybe_get(result, "current_level_index", "level_index")
    if val is None:
        return None
    try:
        return int(val)
    except Exception:
        return None


def _local_available_ids(result: Any, game: Any) -> set[int]:
    raw = maybe_get(result, "available_actions", "actions")
    if raw is None:
        raw = deep_find(result, ("available_actions", "actions"), 2)
    if raw is None:
        raw = maybe_get(game, "_available_actions", "available_actions")
    if raw is None:
        return {i for i in range(1, 8) if ACTION_ID_TO_ENUM.get(i) is not None}
    if isinstance(raw, dict):
        raw = list(raw.keys())
    ids: set[int] = set()
    try:
        iterable = list(raw)
    except Exception:
        iterable = []
    for item in iterable:
        aid = normalize_action_id(item)
        if aid is not None and aid != 0:
            ids.add(int(aid))
    if not ids:
        ids = {i for i in range(1, 8) if ACTION_ID_TO_ENUM.get(i) is not None}
    return ids


def _clone_game(game: Any) -> Any:
    try:
        return copy.deepcopy(game)
    except Exception:
        return copy.deepcopy(game)


class SourcePlanner:
    def __init__(self, game_id: str, source_path: str, class_name: str) -> None:
        self.game_id = str(game_id)
        self.source_path = str(source_path)
        self.class_name = str(class_name)
        self.game_cls = None
        self.loaded = False
        self.fp = FrameProcessor()
        self.solution_cache: dict[int, list[tuple[int, Optional[dict[str, int]]]]] = {}
        self.root_frames: dict[int, np.ndarray] = {}
        self.tracked_paths_by_level: dict[int, list[tuple[Any, ...]]] = {}
        self.transfer_click_points: dict[int, list[tuple[int, int]]] = {}
        self._proposal_cache: dict[tuple[int, str], list[dict[str, Any]]] = {}

    def load(self) -> bool:
        try:
            self.game_cls = _load_game_class(self.source_path, self.class_name)
            self.loaded = self.game_cls is not None
        except Exception as exc:
            self.loaded = False
            PLANNER_LOG.warning("Planner failed to load %s (%s): %s", self.game_id, self.source_path, exc)
        return bool(self.loaded)

    def _new_game(self, level_idx: int) -> tuple[Any, Any, Optional[np.ndarray]]:
        if self.game_cls is None:
            raise RuntimeError("Planner is not loaded.")
        game = self.game_cls()
        if hasattr(game, "set_level"):
            try:
                game.set_level(int(level_idx))
            except Exception:
                pass
        result = None
        frame = None
        for _ in range(3):
            try:
                result = _local_step(game, 0, data=None)
            except Exception:
                result = None
            frame = _local_frame_from_result(result, game)
            if frame is not None:
                break
        return game, result, frame

    def _state_hash(self, game: Any, frame: np.ndarray, tracked_paths: list[tuple[Any, ...]]) -> str:
        frame_hash = self.fp.hash_frame(np.asarray(frame, dtype=np.uint8))
        if not tracked_paths:
            return frame_hash
        extras = []
        for path in tracked_paths:
            value = _get_value_by_path(game, path)
            if value is not None:
                extras.append((_path_text(path), value))
        if not extras:
            return frame_hash
        digest = hashlib.blake2b(repr(tuple(extras)).encode("utf-8"), digest_size=10).hexdigest()
        return frame_hash + "|" + digest

    def _level_solved(self, level_idx: int, result: Any, game: Any) -> bool:
        if _local_state_name(result, game) == "WIN":
            return True
        curr = _local_current_level_index(game, result)
        if curr is not None and curr > int(level_idx):
            return True
        lvl_done = _local_levels_completed(result, game)
        if lvl_done > int(level_idx):
            return True
        return False

    def _game_over(self, result: Any, game: Any) -> bool:
        return _local_state_name(result, game) == "GAME_OVER"

    def _discover_tracked_paths(
        self,
        root_game: Any,
        root_frame: np.ndarray,
        sample_proposals: list[dict[str, Any]],
        level_idx: int,
    ) -> list[tuple[Any, ...]]:
        base_flat = _flatten_small_state(root_game, depth=2, max_items=180)
        if not base_flat:
            return []

        changed: set[tuple[Any, ...]] = set()
        semantic: list[tuple[int, tuple[Any, ...]]] = []
        for path in base_flat.keys():
            score = _semantic_path_score(path)
            if score > 0:
                semantic.append((score, path))
        semantic.sort(key=lambda t: (-t[0], len(t[1]), _path_text(t[1])))

        for proposal in sample_proposals[:18]:
            try:
                g2 = _clone_game(root_game)
                result = _local_step(g2, int(proposal["action_id"]), proposal.get("data"))
                if result is None:
                    continue
                flat2 = _flatten_small_state(g2, depth=2, max_items=180)
            except Exception:
                continue
            for path, v0 in base_flat.items():
                v1 = flat2.get(path)
                if v1 is None:
                    continue
                if v1 != v0:
                    changed.add(path)

        ranked: list[tuple[int, tuple[Any, ...]]] = []
        for path in changed:
            score = 20 + _semantic_path_score(path) - len(path)
            ranked.append((score, path))
        for score, path in semantic:
            if path not in changed:
                ranked.append((3 + score, path))

        ranked.sort(key=lambda t: (-t[0], len(t[1]), _path_text(t[1])))
        out = [path for _, path in ranked[:48]]

        # Drop fields that behave like volatile counters but keep level index / score semantics.
        filtered = []
        for path in out:
            text = _path_text(path)
            if "action_count" in text or "last_action" in text:
                continue
            filtered.append(path)
        return filtered[:40]

    def _segment_click_candidates(self, frame: np.ndarray, game: Any, level_idx: int, max_points: int = 42) -> list[dict[str, Any]]:
        frame = np.asarray(frame, dtype=np.uint8)
        h, w = frame.shape
        bg = int(np.bincount(frame.reshape(-1), minlength=16).argmax())
        scores: dict[tuple[int, int], tuple[float, str]] = {}

        def add_pt(x: int, y: int, score: float, source: str) -> None:
            if not (0 <= int(x) < w and 0 <= int(y) < h):
                return
            key = (int(x), int(y))
            prev = scores.get(key)
            if prev is None or float(score) > float(prev[0]):
                scores[key] = (float(score), str(source))

        for rank, (score, x, y, source) in enumerate(_hidden_coord_points_from_obj(game, frame.shape, max_points=20)):
            add_pt(x, y, 90.0 + float(score) - 0.1 * rank, source)

        transfer_points = self.transfer_click_points.get(int(level_idx), [])
        for j, (x, y) in enumerate(transfer_points[:12]):
            add_pt(x, y, 88.0 - 0.2 * j, "transfer")

        labels, segments = self.fp.segment_frame(frame)
        total_area = int(frame.size)
        ordered_segments = []
        for seg_id, seg in enumerate(segments):
            area = int(seg["area"])
            color = int(seg["color"])
            if color == bg and area > total_area * 0.15:
                continue
            base_score = 72.0 - 6.0 * float(self.fp.segment_group(seg)) - 0.02 * float(area)
            if color != bg:
                base_score += 10.0
            if int(seg.get("number_of_twins", 0)) == 0:
                base_score += 2.0
            ordered_segments.append((base_score, seg_id, seg))
        ordered_segments.sort(key=lambda t: (-t[0], int(t[1])))

        for base_score, seg_id, seg in ordered_segments[:24]:
            reps = self.fp.representative_points(seg)
            x1, y1, x2, y2 = [int(v) for v in seg["bounding_box"]]
            cx = int(round(0.5 * (x1 + x2)))
            cy = int(round(0.5 * (y1 + y2)))

            for rep_rank, (x, y) in enumerate(reps[:3]):
                add_pt(int(x), int(y), base_score - 0.6 * rep_rank, f"segment:{seg_id}")
                for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                    nx, ny = int(x + dx), int(y + dy)
                    if 0 <= nx < w and 0 <= ny < h and int(frame[ny, nx]) == bg:
                        add_pt(nx, ny, base_score - 2.0 - 0.3 * rep_rank, f"boundary:{seg_id}")

            for x, y in [
                (cx, cy),
                (x1 - 1, cy),
                (x2 + 1, cy),
                (cx, y1 - 1),
                (cx, y2 + 1),
                (x1, y1),
                (x2, y1),
                (x1, y2),
                (x2, y2),
            ]:
                add_pt(int(x), int(y), base_score - 2.5, f"bbox:{seg_id}")

        invalid = np.zeros(frame.shape, dtype=bool)
        for (x, y) in scores.keys():
            invalid[int(y), int(x)] = True

        try:
            coarse = self.fp.coarse_grid_points(frame, invalid_mask=invalid, grid_n=5 if max(h, w) >= 32 else 4)
        except Exception:
            coarse = []
        for j, (x, y) in enumerate(coarse[:20]):
            add_pt(int(x), int(y), 36.0 - 0.35 * j, "grid")

        specials = [
            (0, 0), (w - 1, 0), (0, h - 1), (w - 1, h - 1),
            (w // 2, h // 2), (w // 2, 0), (0, h // 2),
            (w - 1, h // 2), (w // 2, h - 1),
        ]
        for j, (x, y) in enumerate(specials):
            add_pt(int(x), int(y), 18.0 - 0.15 * j, "special")

        ordered = sorted(
            [(score, source, x, y) for (x, y), (score, source) in scores.items()],
            key=lambda t: (-t[0], t[3], t[2]),
        )
        out = []
        for score, source, x, y in ordered[:max_points]:
            out.append(
                {
                    "action_id": 6,
                    "data": {"x": int(x), "y": int(y)},
                    "source": str(source),
                    "priority": float(score),
                }
            )
        return out

    def _raw_proposals(
        self,
        game: Any,
        frame: np.ndarray,
        level_idx: int,
        prev_solution: Optional[list[tuple[int, Optional[dict[str, int]]]]] = None,
        depth: int = 0,
        current_path: Optional[list[tuple[int, Optional[dict[str, int]]]]] = None,
    ) -> list[dict[str, Any]]:
        current_path = current_path or []
        avail = _local_available_ids(None, game)
        proposals: list[dict[str, Any]] = []

        preferred_simple = None
        if prev_solution and depth < len(prev_solution):
            if int(prev_solution[depth][0]) != 6:
                preferred_simple = int(prev_solution[depth][0])

        simple_ids = [aid for aid in sorted(avail) if int(aid) != 6]
        if preferred_simple in simple_ids:
            simple_ids = [preferred_simple] + [aid for aid in simple_ids if aid != preferred_simple]

        for aid in simple_ids:
            proposals.append(
                {
                    "action_id": int(aid),
                    "data": None,
                    "source": "simple",
                    "priority": 50.0 + (8.0 if aid == preferred_simple else 0.0),
                }
            )

        if 6 in avail:
            click_props = self._segment_click_candidates(frame, game, level_idx, max_points=42)
            proposals.extend(click_props)

        dedup = []
        seen = set()
        for cand in proposals:
            if cand["data"] is None:
                key = (int(cand["action_id"]), None)
            else:
                key = (int(cand["action_id"]), int(cand["data"]["x"]), int(cand["data"]["y"]))
            if key in seen:
                continue
            seen.add(key)
            dedup.append(cand)
        return dedup

    def _expand_node(
        self,
        game: Any,
        frame: np.ndarray,
        path: list[tuple[int, Optional[dict[str, int]]]],
        tracked_paths: list[tuple[Any, ...]],
        level_idx: int,
        prev_solution: Optional[list[tuple[int, Optional[dict[str, int]]]]] = None,
        max_children: int = 18,
    ) -> list[dict[str, Any]]:
        current_hash = self._state_hash(game, frame, tracked_paths)
        proposals = self._raw_proposals(game, frame, level_idx, prev_solution=prev_solution, depth=len(path), current_path=path)

        children = []
        seen_next_hash: set[str] = set()

        for cand in proposals:
            aid = int(cand["action_id"])
            data = cand.get("data")
            try:
                g2 = _clone_game(game)
                result = _local_step(g2, aid, data=data)
                frame2 = _local_frame_from_result(result, g2)
                if frame2 is None:
                    continue
            except Exception:
                continue

            solved = self._level_solved(level_idx, result, g2)
            game_over = self._game_over(result, g2)
            next_hash = self._state_hash(g2, frame2, tracked_paths)

            if not solved and game_over:
                continue
            if not solved and next_hash == current_hash:
                continue
            if next_hash in seen_next_hash:
                continue
            seen_next_hash.add(next_hash)

            diff = int(np.count_nonzero(np.asarray(frame, dtype=np.uint8) != np.asarray(frame2, dtype=np.uint8)))
            reward = min(6.0, math.log1p(max(0, diff)))
            if diff == 0 and next_hash != current_hash:
                reward += 1.4
            reward += float(cand.get("priority", 0.0)) / 55.0
            if aid != 6:
                reward += 0.15
            if path and aid == int(path[-1][0]) and aid != 6:
                reward -= 0.08
            if solved:
                reward += 1000.0

            new_path = path + [(aid, None if data is None else {"x": int(data["x"]), "y": int(data["y"])})]
            children.append(
                {
                    "game": g2,
                    "frame": frame2,
                    "path": new_path,
                    "hash": next_hash,
                    "step_reward": float(reward),
                    "solved": bool(solved),
                }
            )

        children.sort(key=lambda c: (-float(c["step_reward"]), len(c["path"]), c["hash"]))
        return children[:max_children]

    def _replay_plan(
        self,
        root_game: Any,
        level_idx: int,
        plan: list[tuple[int, Optional[dict[str, int]]]],
    ) -> tuple[bool, list[tuple[int, Optional[dict[str, int]]]], Any, Optional[np.ndarray], bool]:
        g = _clone_game(root_game)
        hist: list[tuple[int, Optional[dict[str, int]]]] = []
        last_frame = _local_frame_from_result(None, g)
        terminal = False
        for aid, data in plan:
            try:
                result = _local_step(g, int(aid), data=data)
                last_frame = _local_frame_from_result(result, g)
                hist.append((int(aid), None if data is None else {"x": int(data["x"]), "y": int(data["y"])}))
            except Exception:
                break
            if self._level_solved(level_idx, result, g):
                return True, hist, g, last_frame, False
            if self._game_over(result, g):
                terminal = True
                break
        return False, hist, g, last_frame, terminal

    def _translate_solution(
        self,
        prev_solution: list[tuple[int, Optional[dict[str, int]]]],
        prev_frame: np.ndarray,
        curr_frame: np.ndarray,
    ) -> list[tuple[int, Optional[dict[str, int]]]]:
        dx, dy = _estimate_click_offset(prev_frame, curr_frame)
        if dx == 0.0 and dy == 0.0:
            return list(prev_solution)
        h, w = curr_frame.shape
        translated = []
        for aid, data in prev_solution:
            if data and "x" in data and "y" in data:
                nd = {
                    "x": int(max(0, min(w - 1, round(float(data["x"]) + dx)))),
                    "y": int(max(0, min(h - 1, round(float(data["y"]) + dy)))),
                }
                translated.append((int(aid), nd))
            else:
                translated.append((int(aid), None if data is None else dict(data)))
        return translated

    def _seed_roots_from_transfer(
        self,
        level_idx: int,
        root_game: Any,
        root_frame: np.ndarray,
        prev_solution: Optional[list[tuple[int, Optional[dict[str, int]]]]],
    ) -> tuple[Optional[list[tuple[int, Optional[dict[str, int]]]]], list[tuple[Any, np.ndarray, list[tuple[int, Optional[dict[str, int]]]], float]]]:
        warm_roots: list[tuple[Any, np.ndarray, list[tuple[int, Optional[dict[str, int]]]], float]] = []
        self.transfer_click_points[int(level_idx)] = []
        if not prev_solution:
            return None, warm_roots

        direct_solved, direct_hist, direct_game, direct_frame, direct_terminal = self._replay_plan(root_game, level_idx, prev_solution)
        if direct_solved:
            return direct_hist, warm_roots
        if direct_hist and direct_frame is not None and not direct_terminal:
            warm_roots.append((direct_game, direct_frame, direct_hist, 0.4 * len(direct_hist)))

        prev_frame = self.root_frames.get(int(level_idx) - 1)
        translated = None
        if prev_frame is not None:
            translated = self._translate_solution(prev_solution, prev_frame, root_frame)
            translated_clicks = []
            for aid, data in translated:
                if int(aid) == 6 and data is not None:
                    translated_clicks.append((int(data["x"]), int(data["y"])))
            self.transfer_click_points[int(level_idx)] = translated_clicks[:12]

            trans_solved, trans_hist, trans_game, trans_frame, trans_terminal = self._replay_plan(root_game, level_idx, translated)
            if trans_solved:
                return trans_hist, warm_roots
            if trans_hist and trans_frame is not None and not trans_terminal:
                warm_roots.append((trans_game, trans_frame, trans_hist, 0.6 * len(trans_hist)))

        return None, warm_roots

    def _short_bfs(
        self,
        root_nodes: list[tuple[Any, np.ndarray, list[tuple[int, Optional[dict[str, int]]]], float]],
        tracked_paths: list[tuple[Any, ...]],
        level_idx: int,
        prev_solution: Optional[list[tuple[int, Optional[dict[str, int]]]]],
        deadline: float,
        max_depth: int,
        max_states: int,
    ) -> Optional[list[tuple[int, Optional[dict[str, int]]]]]:
        queue = deque()
        visited: dict[str, int] = {}
        states_examined = 0

        for game, frame, path, score in root_nodes:
            h = self._state_hash(game, frame, tracked_paths)
            if visited.get(h, 10 ** 9) <= len(path):
                continue
            visited[h] = len(path)
            queue.append((game, frame, path, float(score)))

        while queue and time.time() < deadline and states_examined < max_states:
            game, frame, path, score = queue.popleft()
            if len(path) >= max_depth:
                continue

            children = self._expand_node(
                game,
                frame,
                path,
                tracked_paths,
                level_idx,
                prev_solution=prev_solution,
                max_children=16,
            )
            states_examined += max(1, len(children))
            for child in children:
                if child["solved"]:
                    return child["path"]
            for child in children:
                h = str(child["hash"])
                plen = len(child["path"])
                if visited.get(h, 10 ** 9) <= plen:
                    continue
                visited[h] = plen
                queue.append((child["game"], child["frame"], child["path"], score + float(child["step_reward"])))
        return None

    def _best_first(
        self,
        root_nodes: list[tuple[Any, np.ndarray, list[tuple[int, Optional[dict[str, int]]]], float]],
        tracked_paths: list[tuple[Any, ...]],
        level_idx: int,
        prev_solution: Optional[list[tuple[int, Optional[dict[str, int]]]]],
        deadline: float,
        max_depth: int,
        max_states: int,
        max_heap: int,
    ) -> Optional[list[tuple[int, Optional[dict[str, int]]]]]:
        heap: list[tuple[float, int, int, Any, np.ndarray, list[tuple[int, Optional[dict[str, int]]]], float]] = []
        visited_len: dict[str, int] = {}
        counter = 0

        for game, frame, path, score in root_nodes:
            h = self._state_hash(game, frame, tracked_paths)
            plen = len(path)
            if visited_len.get(h, 10 ** 9) <= plen:
                continue
            visited_len[h] = plen
            priority = -(float(score) - 0.02 * float(plen))
            heapq.heappush(heap, (priority, plen, counter, game, frame, path, float(score)))
            counter += 1

        explored = 0
        while heap and time.time() < deadline and explored < max_states:
            _, _, _, game, frame, path, node_score = heapq.heappop(heap)
            if len(path) >= max_depth:
                continue

            children = self._expand_node(
                game,
                frame,
                path,
                tracked_paths,
                level_idx,
                prev_solution=prev_solution,
                max_children=18,
            )
            explored += max(1, len(children))
            for child in children:
                if child["solved"]:
                    return child["path"]

            for child in children:
                h = str(child["hash"])
                plen = len(child["path"])
                if visited_len.get(h, 10 ** 9) <= plen:
                    continue
                visited_len[h] = plen
                child_score = float(node_score) + float(child["step_reward"])
                priority = -(child_score - 0.025 * float(plen))
                heapq.heappush(heap, (priority, plen, counter, child["game"], child["frame"], child["path"], child_score))
                counter += 1

            if len(heap) > max_heap * 2:
                heap = heapq.nsmallest(max_heap, heap)
                heapq.heapify(heap)

        return None

    def solve_level(
        self,
        level_idx: int,
        prev_solution: Optional[list[tuple[int, Optional[dict[str, int]]]]] = None,
        time_budget_s: float = 20.0,
    ) -> Optional[list[tuple[int, Optional[dict[str, int]]]]]:
        if not self.loaded or self.game_cls is None:
            return None

        deadline = time.time() + max(1.0, float(time_budget_s))
        root_game, root_result, root_frame = self._new_game(level_idx)
        if root_frame is None:
            return None

        self.root_frames[int(level_idx)] = np.asarray(root_frame, dtype=np.uint8).copy()

        raw_samples = self._raw_proposals(root_game, root_frame, level_idx, prev_solution=prev_solution, depth=0, current_path=[])
        tracked_paths = self._discover_tracked_paths(root_game, root_frame, raw_samples, level_idx)
        self.tracked_paths_by_level[int(level_idx)] = tracked_paths

        solved_transfer, warm_transfer_roots = self._seed_roots_from_transfer(level_idx, root_game, root_frame, prev_solution)
        if solved_transfer is not None:
            self.solution_cache[int(level_idx)] = solved_transfer
            return solved_transfer

        root_nodes: list[tuple[Any, np.ndarray, list[tuple[int, Optional[dict[str, int]]]], float]] = [
            (root_game, root_frame, [], 0.0)
        ]
        root_nodes.extend(warm_transfer_roots)

        dedup_roots: list[tuple[Any, np.ndarray, list[tuple[int, Optional[dict[str, int]]]], float]] = []
        seen_hashes: dict[str, int] = {}
        for game, frame, path, score in root_nodes:
            h = self._state_hash(game, frame, tracked_paths)
            if seen_hashes.get(h, 10 ** 9) <= len(path):
                continue
            seen_hashes[h] = len(path)
            dedup_roots.append((game, frame, path, score))
        root_nodes = dedup_roots

        depth_table = {
            0: (10, 20, 18000, 90000, 1200),
            1: (12, 24, 30000, 130000, 1600),
            2: (14, 28, 42000, 180000, 2000),
            3: (16, 32, 65000, 250000, 2600),
            4: (18, 36, 90000, 340000, 3200),
            5: (20, 42, 130000, 460000, 3800),
        }
        short_depth, long_depth, short_states, long_states, max_heap = depth_table.get(
            int(level_idx), (22, 48, 180000, 600000, 4400)
        )

        plan = self._short_bfs(
            root_nodes,
            tracked_paths,
            level_idx,
            prev_solution,
            deadline=min(deadline, time.time() + max(2.0, 0.35 * float(time_budget_s))),
            max_depth=int(short_depth),
            max_states=int(short_states),
        )
        if plan is not None:
            self.solution_cache[int(level_idx)] = plan
            return plan

        if time.time() >= deadline:
            return None

        plan = self._best_first(
            root_nodes,
            tracked_paths,
            level_idx,
            prev_solution,
            deadline=deadline,
            max_depth=int(long_depth),
            max_states=int(long_states),
            max_heap=int(max_heap),
        )
        if plan is not None:
            self.solution_cache[int(level_idx)] = plan
            return plan
        return None


GLOBAL_START = time.time()
GLOBAL_WALLCLOCK_LIMIT_S = 5.55 * 60 * 60

arc = make_arcade()

games_raw = arc.get_environments()
game_ids = normalize_game_ids(games_raw)
if not game_ids:
    raise RuntimeError("No games returned by ARC toolkit / gateway.")

internal_rows: dict[str, dict[str, Any]] = {}
game_summaries: list[dict[str, Any]] = []

for game_index, game_id in enumerate(game_ids):
    if time.time() - GLOBAL_START > GLOBAL_WALLCLOCK_LIMIT_S:
        break

    env = None
    obs = None
    agent = HybridLevelExplorerAgent(game_id, global_memory=GLOBAL_FEATURE_MEMORY)
    actions_in_game = 0
    current_view = None
    completed_flag = False
    current_level_completed = 0
    level_started = True
    planner: Optional[SourcePlanner] = None
    planned_levels_attempted: set[int] = set()
    planner_solved_levels: set[int] = set()

    try:
        env = make_env(arc, game_id)

        try:
            source_path, class_name = _find_game_source_and_class(game_id, env=env)
            if source_path and class_name:
                planner = SourcePlanner(game_id, source_path, class_name)
                if not planner.load():
                    planner = None
        except Exception:
            planner = None

        obs, reset_used = perform_reset(env, reason="initial reset", max_attempts=3)
        actions_in_game += reset_used
        current_level_completed = levels_completed(obs)
        current_view = agent.start_level(obs, env)
        agent.steps_in_level += int(reset_used)
        level_started = True

        while time.time() - GLOBAL_START <= GLOBAL_WALLCLOCK_LIMIT_S:
            st = state_name(obs)
            current_level_completed = levels_completed(obs)
            level_idx = int(current_level_completed)

            if st == "WIN":
                completed_flag = True
                break

            if actions_in_game >= game_budget(level_idx + 1):
                break

            if agent.steps_in_level >= level_budget(level_idx + 1):
                break

            if st == "GAME_OVER":
                if agent.resets_in_level >= 12:
                    break
                obs, used = perform_reset(env, reason="retry level after game over", max_attempts=2)
                actions_in_game += used
                agent.note_external_reset(used)
                current_view = agent._build_view(obs, env)
                level_started = False
                continue

            if level_started and planner is not None and level_idx not in planned_levels_attempted:
                planned_levels_attempted.add(level_idx)
                prev_solution = planner.solution_cache.get(level_idx - 1)
                remaining_global = max(0.0, GLOBAL_WALLCLOCK_LIMIT_S - (time.time() - GLOBAL_START))
                expected_games_left = max(1, len(game_ids) - game_index)
                level_budget_s = planner_time_budget_seconds(level_idx, has_transfer=prev_solution is not None)
                level_budget_s = min(level_budget_s, max(4.0, remaining_global / expected_games_left))

                plan = None
                try:
                    plan = planner.solve_level(level_idx, prev_solution=prev_solution, time_budget_s=level_budget_s)
                except Exception:
                    traceback.print_exc()
                    plan = None

                if plan:
                    replay_solved = False
                    for step_id, (aid, data) in enumerate(plan):
                        reasoning = json.dumps(
                            {
                                "solver": "source_planner",
                                "level_idx": int(level_idx),
                                "step_id": int(step_id),
                                "action_id": int(aid),
                                "data": data,
                            },
                            ensure_ascii=False,
                        )
                        try:
                            obs = take_action(env, int(aid), data=data, reasoning=reasoning)
                            actions_in_game += 1
                            agent.steps_in_level += 1
                        except Exception:
                            obs, used = perform_reset(env, reason="recover after planner replay exception", max_attempts=2)
                            actions_in_game += used
                            agent.note_external_reset(used)
                            current_view = agent._build_view(obs, env)
                            level_started = False
                            break

                        new_state = state_name(obs)
                        new_levels = levels_completed(obs)
                        if new_state == "GAME_OVER":
                            current_view = agent._build_view(obs, env)
                            level_started = False
                            break
                        if new_levels > level_idx or new_state == "WIN":
                            replay_solved = True
                            planner_solved_levels.add(level_idx)
                            if new_state == "WIN":
                                completed_flag = True
                                current_view = None
                                level_started = False
                                break
                            current_view = agent.start_level(obs, env)
                            level_started = True
                            break

                    if completed_flag:
                        break
                    if replay_solved:
                        continue
                    if current_view is None:
                        current_view = agent._build_view(obs, env)

                level_started = False

            if current_view is None:
                current_view = agent._build_view(obs, env)

            choice = agent.choose_candidate(current_view)
            if choice is None:
                break

            if choice["kind"] == "meta_reset":
                obs, used = perform_reset(env, reason=choice.get("reasoning", "strategic reset"), max_attempts=2)
                actions_in_game += used
                agent.note_external_reset(used)
                current_view = agent._build_view(obs, env)
                level_started = False
                continue

            prev_view = current_view
            candidate_idx = int(choice["candidate_idx"])
            reasoning = json.dumps(
                {
                    "candidate_idx": candidate_idx,
                    "kind": choice["kind"],
                    "action_id": int(choice["action_id"]),
                    "tag": list(choice.get("tag", [])) if isinstance(choice.get("tag"), tuple) else choice.get("tag"),
                },
                ensure_ascii=False,
            )

            try:
                obs = take_action(
                    env,
                    int(choice["action_id"]),
                    data=choice.get("data"),
                    reasoning=reasoning,
                )
            except Exception:
                agent.graph.record_test(prev_view.node_id, candidate_idx, -1, None)
                try:
                    agent.remember(prev_view.candidates[candidate_idx], changed=False, errored=True)
                except Exception:
                    pass
                obs, used = perform_reset(env, reason="recover after step exception", max_attempts=2)
                actions_in_game += used
                agent.note_external_reset(used)
                current_view = agent._build_view(obs, env)
                level_started = False
                continue

            actions_in_game += 1

            new_state = state_name(obs)
            new_levels = levels_completed(obs)
            leveled_up = new_levels > prev_view.levels_completed
            game_over = new_state == "GAME_OVER"

            current_view = agent.observe_transition(
                prev_view,
                candidate_idx,
                obs,
                env,
                game_over=game_over,
                level_up=leveled_up,
            )

            if leveled_up:
                current_view = agent.start_level(obs, env)
                level_started = True
            else:
                level_started = False

            if new_state == "WIN":
                completed_flag = True
                break

    except Exception:
        traceback.print_exc()
    finally:
        if env is not None and hasattr(env, "close"):
            try:
                env.close()
            except Exception:
                pass

    final_levels = levels_completed(obs) if obs is not None else current_level_completed
    final_state = state_name(obs) if obs is not None else ""
    completed_flag = completed_flag or final_state == "WIN"

    internal_rows[game_id] = {
        "row_id": f"{game_id}_0",
        "game_id": game_id,
        "end_of_game": bool(completed_flag),
        "score": 0.0,
        "levels_completed": int(final_levels),
        "actions": int(actions_in_game),
        "state": final_state,
        "planner_solved_levels": int(len(planner_solved_levels)),
    }
    game_summaries.append(internal_rows[game_id])

scorecard = None
if hasattr(arc, "close_scorecard"):
    try:
        scorecard = arc.close_scorecard()
    except TypeError:
        try:
            scorecard = arc.close_scorecard(None)
        except Exception:
            scorecard = None
    except Exception:
        scorecard = None

plain_scorecard = to_plain(scorecard)

if isinstance(plain_scorecard, dict):
    env_entries = plain_scorecard.get("environments") or plain_scorecard.get("scores") or []
    for env_item in env_entries:
        if not isinstance(env_item, dict):
            continue
        gid = str(env_item.get("id") or env_item.get("game_id") or "")
        if not gid:
            continue
        row = internal_rows.get(
            gid,
            {
                "row_id": f"{gid}_0",
                "game_id": gid,
                "end_of_game": False,
                "score": 0.0,
            },
        )
        score_val = env_item.get("score", row.get("score", 0.0))
        completed_val = env_item.get("completed", row.get("end_of_game", False))
        try:
            score_val = float(score_val or 0.0)
        except Exception:
            score_val = 0.0
        row["score"] = max(0.0, score_val)
        row["end_of_game"] = bool(completed_val)
        internal_rows[gid] = row

rows = []
for gid in game_ids:
    row = internal_rows.get(
        gid,
        {"row_id": f"{gid}_0", "game_id": gid, "end_of_game": False, "score": 0.0},
    )
    rows.append(
        {
            "row_id": row["row_id"],
            "game_id": row["game_id"],
            "end_of_game": bool(row["end_of_game"]),
            "score": float(row["score"]),
        }
    )

submission = pd.DataFrame(rows, columns=["row_id", "game_id", "end_of_game", "score"])

try:
    maybe_auto = pd.read_parquet(SUBMISSION_PATH)
    if set(["row_id", "game_id", "end_of_game", "score"]).issubset(maybe_auto.columns) and len(maybe_auto) > 1:
        submission = maybe_auto[["row_id", "game_id", "end_of_game", "score"]].copy()
except Exception:
    pass

submission.to_parquet(SUBMISSION_PATH, index=False)
submission = pd.read_parquet(SUBMISSION_PATH)
print(f"saved: {SUBMISSION_PATH}")


2026-03-31 09:12:36 | INFO | Created new scorecard: 62eb082d-3ee2-4210-b7ec-48de3511d38e


INFO:arc_agi.base:Created new scorecard: 62eb082d-3ee2-4210-b7ec-48de3511d38e


2026-03-31 09:12:36 | INFO | Successfully loaded game class Sk48 from /kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/sk48/41055498/sk48.py


INFO:arc_agi.base:Successfully loaded game class Sk48 from /kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/sk48/41055498/sk48.py


2026-03-31 09:12:52 | INFO | Successfully loaded game class Tn36 from /kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/tn36/ab4f63cc/tn36.py


...

2026-03-31 09:20:40 | INFO | Successfully loaded game class G50t from /kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/g50t/5849a774/g50t.py


INFO:arc_agi.base:Successfully loaded game class G50t from /kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/g50t/5849a774/g50t.py


2026-03-31 09:20:55 | INFO | Closed scorecard: 62eb082d-3ee2-4210-b7ec-48de3511d38e


INFO:arc_agi.base:Closed scorecard: 62eb082d-3ee2-4210-b7ec-48de3511d38e


saved: /kaggle/working/submission.parquet


In [3]:
import pandas as pd

if "submission" not in globals():
    submission = pd.read_parquet("/kaggle/working/submission.parquet")

print(submission.head(25))
print("Average score of first 25 games:", float(submission.head(25)["score"].mean()))

             row_id        game_id  end_of_game     score
0   sk48-41055498_0  sk48-41055498        False  0.000000
1   tn36-ab4f63cc_0  tn36-ab4f63cc        False  0.000000
2   m0r0-dadda488_0  m0r0-dadda488        False  0.000000
3   bp35-0a0ad940_0  bp35-0a0ad940        False  0.148633
4   cn04-65d47d14_0  cn04-65d47d14        False  0.000000
5   dc22-4c9bff3e_0  dc22-4c9bff3e        False  0.000000
6   tu93-2b534c15_0  tu93-2b534c15        False  0.000000
7   lp85-305b61c3_0  lp85-305b61c3        False  0.000000
8   ka59-9f096b4a_0  ka59-9f096b4a        False  0.000000
9   wa30-ee6fef47_0  wa30-ee6fef47        False  0.000000
10  vc33-9851e02b_0  vc33-9851e02b        False  1.587302
11  lf52-271a04aa_0  lf52-271a04aa        False  0.000000
12  r11l-aa269680_0  r11l-aa269680        False  0.101273
13  sc25-f9b21a2f_0  sc25-f9b21a2f        False  0.000000
14  sp80-0ee2d095_0  sp80-0ee2d095        False  0.000000
15  ar25-e3c63847_0  ar25-e3c63847        False  0.000000
16  sb26-7fbda